<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L04-agent-to-agent-a2a/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L04-agent-to-agent-a2a/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L04-agent-to-agent-a2a/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L04-agent-to-agent-a2a/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L04 · The Agent2Agent (A2A) protocol, from the wire up

**You will build:** an Agent Card validator, the task lifecycle as a state machine that
refuses illegal moves, an A2A server (`SendMessage`, `GetTask`, `CancelTask`,
`SendStreamingMessage`) and client (polling, streaming, `input-required`), and a conformance
harness built from the specification's own MUST statements, run against your server and
against servers with one planted fault each. Then you measure what a client can learn about
an opaque remote agent (a scripted one, in simulation), against the same capability exposed as
MCP tools.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, no GPU, no network ·
**Prerequisites:** T00-L01-the-8gb-track, T07-L01-agent-loop-from-scratch,
T07-L02-mcp-from-the-wire-up.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `T07-L01`, `T07-L02`

**This lesson assumes you can already:**

- write a state machine in Python as a table of allowed transitions and refuse any move that is not in the table
- read a Server-Sent Events stream: lines that start with data:, with events separated by a blank line
- name a field in nested JSON by its path, such as $.skills[0].id

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Agent2Agent (A2A) Protocol Specification](https://a2a-protocol.org/latest/specification/) · A2A project · *free* — the Agent Card section, TaskState, SendMessage, streaming and the JSON-RPC binding: the MUST statements the exercises and the conformance harness are built from
- [Core Concepts and Components in A2A](https://a2a-protocol.org/latest/topics/key-concepts/) · A2A project · *free* — Agent Card, Task, Message, Artifact and context in plain words, before you meet them in the specification
- [A2A and MCP: Detailed Comparison](https://a2a-protocol.org/latest/topics/a2a-and-mcp/) · A2A project · *free* — why agent-to-agent calls are opaque tasks and not tool calls, the distinction section 14 measures
- [Server-Sent Events (HTML Living Standard, section 9.2)](https://html.spec.whatwg.org/multipage/server-sent-events.html) · WHATWG · *free* — the text/event-stream format, data lines and blank-line dispatch that the streaming exercises fold into a task view
- [MCP specification 2025-11-25: Tools](https://modelcontextprotocol.io/specification/2025-11-25/server/tools) · Model Context Protocol project · *free* — the tool-call shape that section 14 compares with an A2A delegation: typed inputs the client can see

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import copy
import io
import json
import os
import random
import re
import signal
import sys
import threading
import time
import traceback
import uuid
from datetime import datetime, timedelta, timezone
from typing import Any, Callable, Iterator, NamedTuple

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0])

SEED = 20260929          # this lesson's one fixed seed: the demo chain task and every task id
                          # the server hands out derive from it, so re-running prints the same
                          # ids on any machine.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("validate_agent_card",),
    "exercise 2": ("check_transition",),
    "exercise 3": ("apply_history_length", "admit_message", "op_send_message"),
    "exercise 4": ("op_get_task", "op_cancel_task"),
    "exercise 5": ("op_stream_message",),
    "exercise 6": ("poll_until_settled",),
    "exercise 7": ("apply_stream_event",),
    "exercise 8": ("converse",),
    "exercise 9": ("rule_terminal_is_final", "rule_cancel_finished_refused",
                   "rule_stream_is_a_stream", "rule_input_required_continues"),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (apply_history_length, admit_message, op_send_message)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A check that never returns would freeze the notebook (and the grader, which imports it).
# Every check runs under this time limit, far above what a right answer needs.
_CHECK_SECONDS = 10


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


@contextlib.contextmanager
def _time_limit(seconds: float, what: str):
    """Turn a check that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that was already running (the grader's own
    time limit) keeps its deadline if it is due first and is put back, with the time it had
    left, when the check ends. A right answer finishes far inside the limit."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    message = (f"{what} was still running after {seconds:g} s: a loop in your code never ends "
               "(a wait for a state that cannot come, or a count with no limit)")
    started = time.monotonic()
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False,
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"] = owner                               # the frame running the `with`

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in this guard's own
        while frame is not None and frame is not state["owner"]:     # set-up, exit or clean-up (or
            if (frame is state["me"] or frame.f_code in (_fire.__code__, _arm.__code__)    # in code
                    or (frame.f_back is state["owner"] and frame.f_globals is vars(contextlib))):
                return True                              # run from contextlib's __exit__ under the `with`)?
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        if state["next"] is not None and now >= state["next"] - 0.005:
            state["fired"] = True
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            raise _OutOfTime(message, state)
        _arm(now)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        raise AssertionError(message) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        raise AssertionError(message)


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero. A check still running after `_CHECK_SECONDS` is stopped and counts as failed.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What A2A is, and exactly which version this lesson implements

MCP (T07-L02) connects one agent to its **tools**. The Agent2Agent protocol connects an
agent to **another agent** it did not build and cannot inspect: it may run on a different
framework, in a different company, with its own memory and its own tools. The specification
says the two agents collaborate "without needing to share their internal thoughts, plans, or
tool implementations" (`claims.yaml`, `a2a-opaque-execution`). Everything you build here
follows from that one design decision: the wire carries *messages*, *tasks* and *artifacts*,
never a tool call.

**Which version, exactly.** As of 2026-09-29 the specification's own page names `1.0.0` as
its latest released version, and the repository's latest tagged release is a patch of it.
The protocol version on the wire is only the `Major.Minor` part, so this lesson speaks
**A2A 1.0** (`claims.yaml`: `a2a-current-version`, `a2a-version-major-minor`). Version 1.0
changed the wire from the 0.3 you may meet in older tutorials: JSON-RPC methods are
PascalCase (`SendMessage`, not `message/send`) and enums are ProtoJSON names
(`TASK_STATE_INPUT_REQUIRED`, not `input-required`); see `a2a-v03-method-names`,
`a2a-v03-task-states`, `a2a-jsonrpc-binding` and `a2a-protojson-enums`. The specification is
written in layers; this lesson implements one binding, **JSON-RPC 2.0 with Server-Sent-Events
streaming** (`a2a-jsonrpc-protocol`, `a2a-sse-streaming`), over an **in-memory transport**:
real bytes, no sockets, so every run is deterministic and offline.

**What is tested.** The harness in section 13 tests the MUST statements the specification
makes. Where the text lets an agent choose (whether it answers with a `Message` or a
`Task`, whether it accepts a message while a task is still working, whether a stream stays
open at an interrupted state), the lesson makes a choice, says so, and tests nothing about
it. Who runs the standard: Google donated it to the Linux Foundation, a Technical Steering
Committee governs it, and it is licensed Apache-2.0 (`claims.yaml`: `a2a-governance`,
`a2a-governance-tsc`, `a2a-license`). On 27 August 2026 the project announced that it had
been accepted as a Growth Stage project at the Agentic AI Foundation (AAIF), which that
announcement calls Linux Foundation-directed (`a2a-aaif`, `a2a-aaif-lf-directed`).

In [ ]:
A2A_VERSION = "1.0"                    # Major.Minor: patch numbers do not affect compatibility
SUPPORTED_VERSIONS = ("1.0",)          # what this server will speak
WELL_KNOWN_CARD = "/.well-known/agent-card.json"
JSON_CT = "application/json"
SSE_CT = "text/event-stream"

# The task states, spelled exactly as the 1.0 JSON serialisation spells them (ProtoJSON).
SUBMITTED = "TASK_STATE_SUBMITTED"
WORKING = "TASK_STATE_WORKING"
COMPLETED = "TASK_STATE_COMPLETED"
FAILED = "TASK_STATE_FAILED"
CANCELED = "TASK_STATE_CANCELED"
INPUT_REQUIRED = "TASK_STATE_INPUT_REQUIRED"
REJECTED = "TASK_STATE_REJECTED"
AUTH_REQUIRED = "TASK_STATE_AUTH_REQUIRED"
TERMINAL_STATES = frozenset({COMPLETED, FAILED, CANCELED, REJECTED})
INTERRUPTED_STATES = frozenset({INPUT_REQUIRED, AUTH_REQUIRED})
ALL_STATES = TERMINAL_STATES | INTERRUPTED_STATES | {SUBMITTED, WORKING}
ROLE_USER, ROLE_AGENT = "ROLE_USER", "ROLE_AGENT"

# JSON-RPC 2.0's reserved codes, and the A2A-specific ones the specification maps into -32001..-32099.
PARSE_ERROR, INVALID_REQUEST, METHOD_NOT_FOUND = -32700, -32600, -32601
INVALID_PARAMS, INTERNAL_ERROR = -32602, -32603
TASK_NOT_FOUND = -32001            # TaskNotFoundError
TASK_NOT_CANCELABLE = -32002       # TaskNotCancelableError
UNSUPPORTED_OPERATION = -32004     # UnsupportedOperationError
VERSION_NOT_SUPPORTED = -32009     # VersionNotSupportedError
ERROR_NAMES = {PARSE_ERROR: "JSONParseError", INVALID_REQUEST: "InvalidRequestError",
               METHOD_NOT_FOUND: "MethodNotFoundError", INVALID_PARAMS: "InvalidParamsError",
               INTERNAL_ERROR: "InternalError", TASK_NOT_FOUND: "TaskNotFoundError",
               TASK_NOT_CANCELABLE: "TaskNotCancelableError",
               UNSUPPORTED_OPERATION: "UnsupportedOperationError",
               VERSION_NOT_SUPPORTED: "VersionNotSupportedError"}

print(f"this lesson speaks A2A {A2A_VERSION} over JSON-RPC 2.0; "
      f"{len(ALL_STATES)} task states, {len(TERMINAL_STATES)} of them terminal, "
      f"{len(INTERRUPTED_STATES)} interrupted")

## 2. Given: the remote agent, and a copy of T07-L01's tools

The agent behind the wire needs something to *do*, and you already built the something in
T07-L01: a registry of two tools (`lookup`, `submit`) over a synthetic chain of keys, walked
to a final value. The next cell carries that code forward **unchanged** (this lesson never
imports another lesson's files). `ChainJob` is new: the remote agent's *private working*. It
takes one action per server tick: a progress note, a question for the client, an artifact
chunk, or a verdict. It keeps a private log of every internal tool call. That log is the
thing an A2A client never gets to see, and section 14 measures it. Given to you.

In [ ]:
# Given: T07-L01's tool registry and chain generator, carried forward UNCHANGED (same names, same
# bodies: a minimal faithful copy). `TOOL_DESCRIPTIONS` is the one addition: section 14 needs a
# human-readable description per tool to show what an MCP tools/list would publish.
class TransientToolError(Exception):
    """Raised by a tool for a failure worth retrying. Never raised for the caller's own mistake."""


class ToolResult(NamedTuple):
    ok: bool
    output: Any
    error: str | None
    transient: bool


class ToolRegistry(NamedTuple):
    tools: dict[str, Callable[..., Any]]
    schemas: dict[str, dict]


TOOL_SCHEMAS: dict[str, dict] = {
    "lookup": {"type": "object", "properties": {"key": {"type": "string"}},
              "required": ["key"]},
    "submit": {"type": "object", "properties": {"value": {"type": "integer"}},
              "required": ["value"]},
}
# New in this lesson: MCP's tools/list needs a human-readable description per tool, which
# T07-L01's registry never carried (it only needed a machine-checkable schema).
TOOL_DESCRIPTIONS: dict[str, str] = {
    "lookup": "Look up the next key (or the final integer value) for a chain key.",
    "submit": "Submit the final integer value once the chain has been walked to its end.",
}

_ARG_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool,
             "array": list, "object": dict}


def validate_args(schema: dict, args: dict) -> list[str]:
    """Check `args` against a documented JSON Schema subset. From T07-L01, unchanged."""
    errors: list[str] = []
    for name in schema.get("required", []):
        if name not in args:
            errors.append(f"missing required argument {name!r}")
    properties = schema.get("properties", {})
    for name, value in args.items():
        prop = properties.get(name)
        if prop is None:
            continue
        expected = prop.get("type")
        py_type = _ARG_TYPES.get(expected)
        if py_type is not None:
            reject = (isinstance(value, bool) if expected in ("integer", "number")
                      else False)
            if reject or not isinstance(value, py_type):
                errors.append(f"argument {name!r} should be of type {expected!r}, got {value!r}")
                continue
        if "enum" in prop and value not in prop["enum"]:
            errors.append(f"argument {name!r} must be one of {prop['enum']!r}, got {value!r}")
    return errors


def call_tool(registry: ToolRegistry, name: str, args: dict, validate: bool = True) -> ToolResult:
    """Execute one tool call through `registry`, never letting an exception escape. From
    T07-L01, unchanged."""
    if name not in registry.tools:
        return ToolResult(ok=False, output=None, error=f"unknown tool {name!r}", transient=False)
    schema = registry.schemas.get(name, {})
    if validate:
        errors = validate_args(schema, args)
        if errors:
            return ToolResult(ok=False, output=None,
                              error="invalid arguments: " + "; ".join(errors), transient=False)
        try:
            return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                              transient=False)
        except TransientToolError as exc:
            return ToolResult(ok=False, output=None, error=str(exc), transient=True)
        except Exception as exc:
            return ToolResult(ok=False, output=None,
                              error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                              transient=False)
    try:
        return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                          transient=False)
    except Exception as exc:
        return ToolResult(ok=False, output=None,
                          error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                          transient=True)


class ChainTask(NamedTuple):
    task_id: int
    start_key: str
    key_sequence: list[str]
    chain: dict[str, Any]
    target: int
    length: int


def generate_chain_task(length: int, seed: int) -> ChainTask:
    """One deterministic lookup chain of the given length. From T07-L01, unchanged."""
    rng = random.Random(seed)
    keys = [f"k{i}_{rng.randrange(1_000_000):06d}" for i in range(length)]
    chain: dict[str, Any] = {keys[i]: keys[i + 1] for i in range(length - 1)}
    target = rng.randint(100, 999)
    chain[keys[-1]] = target
    return ChainTask(task_id=seed, start_key=keys[0], key_sequence=keys, chain=chain,
                     target=target, length=length)


def make_registry(chain: dict, target: int, transient_rate: float,
                  rng: random.Random) -> ToolRegistry:
    """A registry over ONE task's chain. From T07-L01, unchanged."""
    def lookup(key):
        if rng.random() < transient_rate:
            raise TransientToolError("lookup timed out")
        if key not in chain:
            raise ValueError(f"unknown key {key!r}")
        val = chain[key]
        return {"next_key": val} if isinstance(val, str) else {"value": val}

    def submit(value):
        return {"correct": value == target}

    return ToolRegistry(tools={"lookup": lookup, "submit": submit}, schemas=TOOL_SCHEMAS)



In [ ]:
# Given: the remote agent's private working. Nothing in a ChainJob is ever put on the wire
# directly: a client only sees the messages, artifacts and states the SERVER chooses to publish.
class Action(NamedTuple):
    """One thing the agent does per tick. kind: progress | ask | artifact | done | fail | reject."""
    kind: str
    text: str = ""
    artifact_id: str = ""
    name: str = ""
    append: bool = False
    last: bool = False


_KEY = re.compile(r"k\d+_\d{6}")


class ChainJob:
    """Solves 'follow the lookup chain from a start key'. One `step()` per server tick.

    Ask it something that is not about a chain and it rejects the task; ask about a chain
    without naming a start key and it asks which one; name a key it does not know and the task
    fails. `verbose=True` makes the agent chatty: its progress notes then name the keys it
    looked up. Whether a remote agent says such things is ITS choice; the protocol has no
    field that obliges it to (section 14 measures the difference).
    """

    def __init__(self, registry: ToolRegistry, verbose: bool = False):
        self.registry, self.verbose = registry, verbose
        self.calls: list[tuple[str, dict]] = []      # PRIVATE log of internal tool calls
        self.phase, self.text = "new", ""
        self.cursor: str | None = None
        self.value: int | None = None
        self.hops = 0

    def feed(self, text: str) -> None:
        """A client message arrived: remember its words and any chain key it names."""
        self.text = text
        found = _KEY.search(text)
        if found:
            self.cursor = found.group(0)

    def _call(self, name: str, args: dict) -> ToolResult:
        self.calls.append((name, dict(args)))
        return call_tool(self.registry, name, args)

    def step(self) -> Action:
        if self.phase == "new":
            if self.cursor is None and "chain" not in self.text.lower():
                self.phase = "over"
                return Action("reject", "I only solve lookup chains.")
            self.phase = "start"
            return Action("progress", "started")
        if self.phase == "start":
            if self.cursor is None:
                self.phase = "waiting"
                return Action("ask", "Which start key should I begin the chain from?")
            self.phase = "walk"
        if self.phase == "waiting":
            if self.cursor is None:
                return Action("ask", "I still need a start key: which one?")
            self.phase = "walk"
        if self.phase == "walk":
            here = self.cursor
            result = self._call("lookup", {"key": here})
            if not result.ok:
                self.phase = "over"
                return Action("fail", "That start key is not one I know.")
            self.hops += 1
            if "next_key" in result.output:
                self.cursor = result.output["next_key"]
                note = (f"looked up {here}, next is {self.cursor}" if self.verbose
                        else f"hop {self.hops} done")
                return Action("progress", note)
            self.value, self.phase = result.output["value"], "answer1"
            return Action("progress", f"looked up {here}, reached the end" if self.verbose
                          else f"hop {self.hops} done")
        if self.phase == "answer1":
            self._call("submit", {"value": self.value})
            self.phase = "answer2"
            return Action("artifact", "The chain ends at ", "answer-1", "answer", False, False)
        if self.phase == "answer2":
            self.phase = "finish"
            return Action("artifact", str(self.value), "answer-1", "answer", True, True)
        self.phase = "over"
        return Action("done")


# The one task this whole lesson's demos walk. The remote agent's job, and every probe below,
# is built from this same constant and the fixed SEED, so nothing has to be transmitted.
_DEMO_TASK = generate_chain_task(length=3, seed=SEED)
print(f"the demo task: start at {_DEMO_TASK.start_key!r}, chain length {_DEMO_TASK.length}")


def make_job(verbose: bool = False) -> ChainJob:
    """A fresh agent working on the demo chain. Every server below is handed this factory."""
    registry = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
    return ChainJob(registry, verbose=verbose)

## 3. Exercise 1: the Agent Card validator

Before anything else, an A2A client fetches the server's **Agent Card**: a JSON document that
says who the agent is, which interfaces it speaks (URL, binding, protocol version), which
optional capabilities it has, and its skills. The specification says a server MUST make one
available, at a well-known URI among other ways (`claims.yaml`, `a2a-card-must-be-available`).
It is defined in the protobuf file; a field marked REQUIRED "MUST be present and set", and
**a required array MUST contain at least one element** (`a2a-required-fields`).

Write `validate_agent_card`. It returns a list of problems, each starting with the path of the
offending field, then a colon (`skills[0].tags: ...`), and an empty list for a valid card.
The required fields are listed in its docstring. Optional fields that are absent are fine,
unknown extra fields are ignored (the specification says implementations SHOULD ignore
unrecognised fields), and `capabilities` is required but **may be an empty object**.

<details><summary>💡 Hint 1 — what to think about</summary>

Four kinds of problem repeat: a required string that is missing or empty, a required array
that is missing or empty, a value of the wrong type, and a nested object (an interface, a
skill) that has its own required fields. Which small helper could you write once and call for
each of them, passing the path so the message names the field?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Collect problems in a list and never stop at the first one. For `capabilities`, ask whether
the value is an object at all, not whether it is empty: an empty object is a perfectly good
capabilities declaration. For each array, report a missing-or-empty array once, then walk the
elements that are there and check each one, building paths like `skills[1].tags[0]`.
</details>

In [ ]:
def make_agent_card(streaming: bool = True) -> dict:
    """The card our chain-solving agent publishes. Given."""
    return {
        "name": "Chain Solver",
        "description": "Follows a chain of lookups from a start key to the value at its end.",
        "supportedInterfaces": [{"url": "https://chain-agent.example/a2a",
                                 "protocolBinding": "JSONRPC", "protocolVersion": A2A_VERSION}],
        "provider": {"organization": "Synapsa Commons (teaching example)",
                     "url": "https://synapsa.realai.eu"},
        "version": "1.0.0",
        "capabilities": {"streaming": streaming, "pushNotifications": False},
        "defaultInputModes": ["text/plain"],
        "defaultOutputModes": ["text/plain"],
        "skills": [{"id": "solve-lookup-chain", "name": "Solve a lookup chain",
                    "description": "Given a start key, follows the chain and returns its end value.",
                    "tags": ["lookup", "chain"],
                    "examples": ["Solve the lookup chain from k0_012345"]}],
    }


def validate_agent_card(card: Any) -> list[str]:
    """Check an Agent Card against the 1.0 required-field rules.

    Returns one string per problem, each of the form "<path>: <what is wrong>", in any order;
    an empty list means the card is valid. A path is the field's JSON path with `[i]` for
    array elements, e.g. `skills[0].tags`. A card that is not an object at all is the single
    problem `"card: must be a JSON object"`.

    Required at the top level: `name`, `description`, `version` (non-empty strings);
    `supportedInterfaces`, `defaultInputModes`, `defaultOutputModes`, `skills` (arrays with at
    least one element); `capabilities` (an object, possibly empty). Every interface needs
    non-empty `url`, `protocolBinding`, `protocolVersion`; every skill needs non-empty `id`,
    `name`, `description` and a non-empty `tags` array of strings; the modes are non-empty
    strings. If `provider` is present it needs non-empty `url` and `organization`. If any of
    `capabilities.streaming`, `.pushNotifications`, `.extendedAgentCard` is present it must be
    a boolean.

    Example:
        >>> card = make_agent_card(); del card["skills"]
        >>> [problem.split(":")[0] for problem in validate_agent_card(card)]
        ['skills']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_validate_agent_card() -> None:
    good = validate_agent_card(make_agent_card())
    assert good == [], f"the lesson's own valid card was rejected: {good!r}"
    assert validate_agent_card("nope") == ["card: must be a JSON object"], (
        "a card that is not a JSON object must be the single problem 'card: must be a JSON object'")

    def paths(card) -> set[str]:
        return {problem.split(":")[0] for problem in validate_agent_card(card)}

    for key in ("name", "description", "version", "supportedInterfaces", "capabilities",
                "defaultInputModes", "defaultOutputModes", "skills"):
        broken = make_agent_card()
        del broken[key]
        assert key in paths(broken), (
            f"a card with no {key!r} was not flagged under the path {key!r}; report every missing "
            "required field, each starting with its own path")
    empty_skills = make_agent_card()
    empty_skills["skills"] = []
    assert "skills" in paths(empty_skills), (
        "'skills: []' passed: a required array MUST contain at least one element")
    empty_tags = make_agent_card()
    empty_tags["skills"][0]["tags"] = []
    assert "skills[0].tags" in paths(empty_tags), (
        "a skill with an empty tags array passed; tags is a required array, so it needs an "
        "element, and the path must be 'skills[0].tags'")
    empty_caps = make_agent_card()
    empty_caps["capabilities"] = {}
    assert validate_agent_card(empty_caps) == [], (
        f"an EMPTY capabilities object is valid (every capability inside it is optional): "
        f"{validate_agent_card(empty_caps)!r}; test whether it is an object, not whether it is truthy")
    extra = make_agent_card()
    extra["x-vendor-note"] = {"anything": True}
    assert validate_agent_card(extra) == [], (
        "an unknown extra field made the card invalid; the specification says to ignore them")
    bad_flag = make_agent_card()
    bad_flag["capabilities"]["streaming"] = "yes"
    assert "capabilities.streaming" in paths(bad_flag), "a non-boolean 'streaming' passed"
    bad_iface = make_agent_card()
    del bad_iface["supportedInterfaces"][0]["protocolVersion"]
    assert "supportedInterfaces[0].protocolVersion" in paths(bad_iface), (
        "every interface needs url, protocolBinding and protocolVersion, each reported with its "
        "own indexed path")
    blank_name = make_agent_card()
    blank_name["name"] = ""
    assert "name" in paths(blank_name), "an empty-string name is 'not set'; it must be flagged"
    two = make_agent_card()
    del two["name"]
    del two["version"]
    assert {"name", "version"} <= paths(two), (
        "only one problem was reported for a card with two; collect every problem, do not stop "
        "at the first")
    print("exercise 1 looks right: the validator names every violation under its own path")


_try("exercise 1", _check_validate_agent_card)

## 4. Exercise 2: the task lifecycle as a state machine

A message that starts work becomes a **Task**, and a task moves through states. The
specification names the states, marks four of them **terminal** and two **interrupted**, and
says that once a task is terminal "it cannot restart": any follow-up is a new task
(`claims.yaml`, `a2a-terminal-immutable`, `a2a-task-states`). It does *not* publish a full
transition graph (`a2a-no-transition-table`). The table below is therefore this lesson's own
conservative reading. Only the first rule is the specification's; the rest are our choices,
and the harness in section 13 tests only what the specification itself demands.

Write `check_transition(current, new)`. It returns `None` when the move is allowed and
raises `IllegalTransition` when it is not:

* both names must be real states (`TASK_STATE_UNSPECIFIED` is not one), else illegal;
* a **terminal** state can never be left, not even to itself (the specification's rule);
* nothing ever moves *into* `SUBMITTED`; it is only how a task starts;
* `SUBMITTED` may move to `WORKING`, `REJECTED`, `CANCELED` or `FAILED`;
* `WORKING` may stay `WORKING` (a progress update) or move to any of `COMPLETED`, `FAILED`,
  `CANCELED`, `REJECTED`, `INPUT_REQUIRED`, `AUTH_REQUIRED`;
* an **interrupted** state may move to `WORKING` (the client answered), `CANCELED`, `FAILED`
  or `REJECTED`, and never to itself and never straight to `COMPLETED`.

<details><summary>💡 Hint 1 — what to think about</summary>

The rules above are a table: for each state, the set of states it may move to. What is the
allowed set of a terminal state? If you look a state up in a dictionary and it is not there,
what should that mean?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Reject unknown names first, so a typo cannot slip through as "not in the table, so nothing
to check". Then look up the allowed set for `current` (an empty set when it has none) and
raise if `new` is not in it. The message should name the states involved.
</details>

In [ ]:
class IllegalTransition(Exception):
    """Raised by check_transition for a move the lifecycle does not allow."""


def check_transition(current: str, new: str) -> None:
    """Return None if a task may move from `current` to `new`; raise IllegalTransition if not.

    Example:
        >>> check_transition(WORKING, INPUT_REQUIRED) is None
        True
        >>> check_transition(COMPLETED, WORKING)
        Traceback (most recent call last):
            ...
        IllegalTransition: TASK_STATE_COMPLETED -> TASK_STATE_WORKING is not allowed
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_check_transition() -> None:
    assert check_transition(SUBMITTED, WORKING) is None, "SUBMITTED -> WORKING must be allowed"
    assert check_transition(WORKING, COMPLETED) is None, "WORKING -> COMPLETED must be allowed"
    assert check_transition(WORKING, WORKING) is None, (
        "WORKING -> WORKING must be allowed: a progress update keeps the state")
    assert check_transition(WORKING, INPUT_REQUIRED) is None, (
        "WORKING -> INPUT_REQUIRED must be allowed: it is how the agent asks the client a question")
    assert check_transition(INPUT_REQUIRED, WORKING) is None, (
        "INPUT_REQUIRED -> WORKING must be allowed: it is how the task resumes after the answer")

    def refused(a: str, b: str) -> bool:
        try:
            check_transition(a, b)
        except IllegalTransition:
            return True
        return False

    for done in sorted(TERMINAL_STATES):
        for target in sorted(ALL_STATES):
            assert refused(done, target), (
                f"{done} -> {target} was accepted: a terminal state can never be left, not even "
                "to itself; a follow-up to a finished task is a NEW task")
    assert refused(WORKING, SUBMITTED) and refused(INPUT_REQUIRED, SUBMITTED), (
        "nothing may move INTO SUBMITTED")
    assert refused(INPUT_REQUIRED, COMPLETED), (
        "INPUT_REQUIRED -> COMPLETED was accepted: the task must go back to WORKING first")
    assert refused(INPUT_REQUIRED, INPUT_REQUIRED), "an interrupted state does not move to itself"
    assert refused(SUBMITTED, INPUT_REQUIRED) and refused(SUBMITTED, COMPLETED), (
        "SUBMITTED must go through WORKING (or be rejected, canceled or failed)")
    assert refused("TASK_STATE_UNSPECIFIED", WORKING) and refused(WORKING, "TASK_STATE_UNSPECIFIED"), (
        "TASK_STATE_UNSPECIFIED is not a state a task can be in or move to")
    assert refused(WORKING, "working"), "a name that is not a real state must be refused"
    print("exercise 2 looks right: legal moves pass, everything else raises IllegalTransition")


_try("exercise 2", _check_check_transition)

## 5. Given: the server around your handlers

The next cell is the server plumbing you do **not** write: the JSON-RPC envelope (parse error,
invalid request, version check, method lookup, SSE framing for streams), the task store, the
logical clock (a tick per unit of work; **no wall clock anywhere**), and `enter_state`, which
moves a task through *your* `check_transition`. Your handlers in exercises 3 to 5 are plugged in
through `methods()`. Read `handle` and `enter_state` before you write: they are the contract
your handlers run inside. Two wire facts to keep in view: `SendMessage` answers with an object
holding exactly one of `task` or `message`, while `GetTask` and `CancelTask` answer with the
Task itself; and a streamed reply is a list of events, each one `data: <JSON-RPC response>`
followed by a blank line.

In [ ]:
# A loop that waits for something that will never happen (a state a settled task cannot reach on
# its own, a poll with no limit) must FAIL with a name, not freeze the notebook. No correct
# solution comes near this limit: the whole lesson sends at most a few dozen requests to a server.
RUNAWAY_LIMIT = 500


class RunawayLoop(Exception):
    """Raised when code keeps calling a server (or a scripted peer) far past any sensible count."""


class A2AError(Exception):
    """An error the server answers with as a JSON-RPC error object."""

    def __init__(self, code: int, message: str, data: list | None = None):
        super().__init__(message)
        self.code, self.message, self.data = code, message, data


class A2AWireError(Exception):
    """The peer broke a wire rule (client side): a wrong content type, a wrong id, a bad event."""


class A2ARemoteError(Exception):
    """The peer answered with a JSON-RPC error object (client side)."""

    def __init__(self, code: int, message: str):
        super().__init__(f"{ERROR_NAMES.get(code, 'error')} ({code}): {message}")
        self.code, self.message = code, message


class Reply(NamedTuple):
    """What comes back from one in-memory request: a status, a content type, and the body as
    a tuple of chunks (one chunk for JSON; one chunk per event for a stream)."""
    status: int
    content_type: str
    chunks: tuple[bytes, ...]


def _dumps(obj: Any) -> bytes:
    return json.dumps(obj, separators=(",", ":"), ensure_ascii=False).encode("utf-8")


def message_text(message: dict | None) -> str:
    """The text parts of a Message, joined by newlines (empty for None)."""
    if not message:
        return ""
    return "\n".join(p["text"] for p in message.get("parts", []) if "text" in p)


def message_problem(message: Any) -> str | None:
    """What is wrong with a client Message, or None. `messageId`, `role` (ROLE_USER) and a
    non-empty `parts` array are required, and each Part must hold exactly one of `text`, `raw`,
    `url`, `data` (claims.yaml, a2a-part-exactly-one)."""
    if not isinstance(message, dict):
        return "must be an object"
    if not isinstance(message.get("messageId"), str) or not message["messageId"]:
        return "messageId is required"
    if message.get("role") != ROLE_USER:
        return f"role must be {ROLE_USER}"
    parts = message.get("parts")
    if not isinstance(parts, list) or not parts:
        return "parts must be a non-empty array"
    for i, part in enumerate(parts):
        content = [k for k in ("text", "raw", "url", "data") if isinstance(part, dict) and k in part]
        if len(content) != 1:
            return f"parts[{i}] must contain exactly one of text, raw, url, data"
    return None


def read_history_length(params: dict) -> int | None:
    """`historyLength` from a params object (or its `configuration`): None if unset, else a
    non-negative integer. Raises INVALID_PARAMS for anything else (a bool is not an integer).
    The specification says servers MUST validate input parameters before processing."""
    value = params.get("historyLength")
    if value is None:
        return None
    if isinstance(value, bool) or not isinstance(value, int) or value < 0:
        raise A2AError(INVALID_PARAMS, "Invalid parameters: historyLength must be a "
                                       "non-negative integer")
    return value


class TaskRecord:
    """Everything the server keeps about one task. Private: clients only ever see `task_view`."""

    def __init__(self, task_id: str, context_id: str, job: ChainJob):
        self.id, self.context_id, self.job = task_id, context_id, job
        self.state = SUBMITTED
        self.status_message: dict | None = None
        self.stamp = ""
        self.artifacts: dict[str, dict] = {}
        self.history: list[dict] = []
        self.idle_ticks = 0                              # ticks received while already settled


class A2AServer:
    """An A2A 1.0 server (JSON-RPC binding) over an in-memory transport.

    `job_factory` builds the remote agent's private working for each new task. `streaming`
    is the capability the Agent Card declares. Handlers are the module-level `op_*` functions
    you write; `methods()` maps wire method names to them.
    """

    def __init__(self, job_factory: Callable[[], ChainJob], streaming: bool = True, seed: int = 0):
        self.job_factory, self.streaming = job_factory, streaming
        self.tasks: dict[str, TaskRecord] = {}
        self.stream_methods = {"SendStreamingMessage"}
        self._rng = random.Random(SEED + seed)
        self.clock = 0
        self.requests, self.idle_advances = 0, 0          # for the RunawayLoop guard only

    # -- identifiers and time (deterministic: a seeded generator and a logical clock) ------
    def new_id(self) -> str:
        return str(uuid.UUID(int=self._rng.getrandbits(128), version=4))

    def now(self) -> str:
        """An ISO 8601 UTC timestamp, millisecond precision, derived from a logical clock."""
        self.clock += 1
        when = datetime(2026, 1, 1, tzinfo=timezone.utc) + timedelta(milliseconds=250 * self.clock)
        return when.strftime("%Y-%m-%dT%H:%M:%S.") + f"{when.microsecond // 1000:03d}Z"

    # -- discovery ---------------------------------------------------------------------------
    def agent_card(self) -> dict:
        return make_agent_card(streaming=self.streaming)

    def get(self, path: str) -> Reply:
        """The HTTP GET side: the Agent Card at its well-known URI, 404 for anything else."""
        if path == WELL_KNOWN_CARD:
            return Reply(200, JSON_CT, (_dumps(self.agent_card()),))
        return Reply(404, JSON_CT, (_dumps({"error": "not found"}),))

    # -- the task store ---------------------------------------------------------------------
    def new_task(self, message: dict) -> TaskRecord:
        """Create a task from a client message: server-generated id, context preserved or new."""
        task_id = self.new_id()
        context_id = message.get("contextId") or self.new_id()
        rec = TaskRecord(task_id, context_id, self.job_factory())
        rec.stamp = self.now()
        rec.history.append(dict(message, taskId=task_id, contextId=context_id))
        rec.job.feed(message_text(message))
        self.tasks[task_id] = rec
        return rec

    def deliver(self, rec: TaskRecord, message: dict) -> None:
        """A follow-up message for an existing task. An answer to an interrupted task resumes it."""
        rec.history.append(dict(message, taskId=rec.id, contextId=rec.context_id))
        rec.job.feed(message_text(message))
        if rec.state in INTERRUPTED_STATES:
            self.enter_state(rec, WORKING)

    def agent_message(self, rec: TaskRecord, text: str) -> dict:
        return {"messageId": self.new_id(), "role": ROLE_AGENT, "parts": [{"text": text}],
                "taskId": rec.id, "contextId": rec.context_id}

    def status_of(self, rec: TaskRecord) -> dict:
        status: dict = {"state": rec.state}
        if rec.status_message:
            status["message"] = copy.deepcopy(rec.status_message)
        status["timestamp"] = rec.stamp
        return status

    def enter_state(self, rec: TaskRecord, new_state: str, text: str | None = None) -> dict:
        """Move a task to `new_state` through YOUR check_transition; returns the statusUpdate event."""
        try:
            check_transition(rec.state, new_state)
        except IllegalTransition as exc:
            raise A2AError(INTERNAL_ERROR, f"internal error: {exc}") from None
        rec.state = new_state
        rec.status_message = self.agent_message(rec, text) if text else None
        if rec.status_message:
            rec.history.append(rec.status_message)
        rec.stamp = self.now()
        return {"statusUpdate": {"taskId": rec.id, "contextId": rec.context_id,
                                 "status": self.status_of(rec)}}

    def tick(self, rec: TaskRecord) -> list[dict]:
        """One unit of work on one task; returns the StreamResponse events it produced."""
        if rec.state in TERMINAL_STATES or rec.state in INTERRUPTED_STATES:
            rec.idle_ticks += 1
            if rec.idle_ticks > RUNAWAY_LIMIT:
                raise RunawayLoop(f"tick() was called {RUNAWAY_LIMIT} times on a task already in "
                                  f"{rec.state}: a loop is waiting for a state this task cannot reach "
                                  "on its own (settled means terminal OR interrupted)")
            return []
        act = rec.job.step()
        if act.kind == "progress":
            return [self.enter_state(rec, WORKING, act.text)]
        if act.kind == "ask":
            return [self.enter_state(rec, INPUT_REQUIRED, act.text)]
        if act.kind == "fail":
            return [self.enter_state(rec, FAILED, act.text)]
        if act.kind == "reject":
            return [self.enter_state(rec, REJECTED, act.text)]
        if act.kind == "done":
            return [self.enter_state(rec, COMPLETED)]
        part = {"text": act.text}
        if act.append and act.artifact_id in rec.artifacts:
            rec.artifacts[act.artifact_id]["parts"].append(part)
        else:
            rec.artifacts[act.artifact_id] = {"artifactId": act.artifact_id, "name": act.name,
                                              "parts": [part]}
        rec.stamp = self.now()
        chunk = {"artifactId": act.artifact_id, "name": act.name, "parts": [dict(part)]}
        return [{"artifactUpdate": {"taskId": rec.id, "contextId": rec.context_id,
                                    "artifact": chunk, "append": act.append,
                                    "lastChunk": act.last}}]

    def run_until_settled(self, rec: TaskRecord, limit: int = 200) -> list[dict]:
        """Tick until the task is terminal or interrupted; returns every event produced, in order."""
        events: list[dict] = []
        for _ in range(limit):
            if rec.state in TERMINAL_STATES or rec.state in INTERRUPTED_STATES:
                return events
            events += self.tick(rec)
        raise A2AError(INTERNAL_ERROR, "internal error: the task did not settle")

    def advance(self, ticks: int = 1) -> None:
        """Let time pass: one tick for every task that is still running."""
        for _ in range(ticks):
            running = [rec for rec in self.tasks.values() if rec.state in (SUBMITTED, WORKING)]
            self.idle_advances = 0 if running else self.idle_advances + 1
            if self.idle_advances > RUNAWAY_LIMIT:
                raise RunawayLoop(f"advance() was called {RUNAWAY_LIMIT} times in a row with no task "
                                  "running: a loop is waiting for a state no task will reach by itself")
            for rec in running:
                self.tick(rec)

    def task_view(self, rec: TaskRecord) -> dict:
        """The Task object a client sees: a fresh copy, with the FULL history."""
        task: dict = {"id": rec.id, "contextId": rec.context_id, "status": self.status_of(rec)}
        if rec.artifacts:
            task["artifacts"] = copy.deepcopy(list(rec.artifacts.values()))
        if rec.history:
            task["history"] = copy.deepcopy(rec.history)
        return task

    # -- the JSON-RPC layer -------------------------------------------------------------------
    def methods(self) -> dict[str, Callable[["A2AServer", dict], Any]]:
        return {"SendMessage": op_send_message, "SendStreamingMessage": op_stream_message,
                "GetTask": op_get_task, "CancelTask": op_cancel_task}

    def check_version(self, headers: dict | None) -> None:
        """The A2A-Version service parameter: an empty value means 0.3; only Major.Minor counts."""
        raw = next((v for k, v in (headers or {}).items() if k.lower() == "a2a-version"), "")
        want = ".".join(raw.strip().split(".")[:2]) or "0.3"
        if want not in SUPPORTED_VERSIONS:
            raise A2AError(VERSION_NOT_SUPPORTED, "Protocol Version Not Supported",
                           [{"@type": "type.googleapis.com/google.rpc.ErrorInfo",
                             "reason": "VERSION_NOT_SUPPORTED", "domain": "a2a-protocol.org",
                             "metadata": {"requested": want,
                                          "supported": ",".join(SUPPORTED_VERSIONS)}}])

    def _error(self, rid: Any, code: int, message: str, data: list | None = None) -> Reply:
        error: dict = {"code": code, "message": message}
        if data:
            error["data"] = data
        return Reply(200, JSON_CT, (_dumps({"jsonrpc": "2.0", "id": rid, "error": error}),))

    def handle(self, raw: bytes, headers: dict | None = None) -> Reply:
        """One POST: bytes in, a Reply out. Only A2AError becomes an error response: a bug in a
        handler (including an unfinished exercise) is NOT hidden from you."""
        self.requests += 1
        if self.requests > RUNAWAY_LIMIT:
            raise RunawayLoop(f"this server has been sent {RUNAWAY_LIMIT} requests: a loop keeps asking "
                              "(a poll with no limit, or a wait for a state that never comes)")
        try:
            request = json.loads(raw.decode("utf-8"))
        except (UnicodeDecodeError, ValueError):
            return self._error(None, PARSE_ERROR, "Invalid JSON payload")
        rid = request.get("id") if isinstance(request, dict) else None
        if (not isinstance(request, dict) or request.get("jsonrpc") != "2.0"
                or not isinstance(request.get("method"), str)
                or isinstance(rid, bool) or not isinstance(rid, (str, int))):
            return self._error(None, INVALID_REQUEST, "Request payload validation error")
        try:
            self.check_version(headers)
            handler = self.methods().get(request["method"])
            if handler is None:
                raise A2AError(METHOD_NOT_FOUND, "Method not found")
            params = request.get("params", {})
            if not isinstance(params, dict):
                raise A2AError(INVALID_PARAMS, "Invalid parameters: params must be an object")
            result = handler(self, params)
        except A2AError as exc:
            return self._error(rid, exc.code, exc.message, exc.data)
        if request["method"] in self.stream_methods:
            return Reply(200, SSE_CT, tuple(
                b"data: " + _dumps({"jsonrpc": "2.0", "id": rid, "result": event}) + b"\n\n"
                for event in result))
        return Reply(200, JSON_CT, (_dumps({"jsonrpc": "2.0", "id": rid, "result": result}),))



## 6. Exercise 3: accepting a message, and `SendMessage`

`SendMessage` is the front door. A client message either **starts a task** (no `taskId`) or
**continues one** (a `taskId`, and optionally the `contextId`). The specification's rules
for the second case are where a server goes wrong (`claims.yaml`: `a2a-task-not-found`,
`a2a-context-mismatch`, `a2a-terminal-rejects-messages`, `a2a-validate-input`):

* a `taskId` that names no task is `TaskNotFoundError` (-32001), and an *empty string* is still
  a `taskId` that names no task;
* a `contextId` that differs from the task's own is rejected (the specification names no code;
  this lesson uses `INVALID_PARAMS`);
* a message to a task in a **terminal** state is `UnsupportedOperationError` (-32004), and the
  refused message must leave no trace on the task;
* an invalid message (see `message_problem`) or `historyLength` is `INVALID_PARAMS`, and
  **no task may be created for it**.

Three functions. `apply_history_length` implements `historyLength` (`a2a-history-length`):
unset means no limit, **`0` means none** (omit the `history` key), and `n > 0` means at most
the last `n` messages. `admit_message(server, params)` does the admission above and returns
the task record (use `server.new_task` and `server.deliver`). `op_send_message` calls it and
answers `{"task": <Task>}`: **by default the call blocks until the task is terminal or
interrupted** (`a2a-send-blocking`); with `configuration.returnImmediately` true it returns as
soon as the task exists (`a2a-send-return-immediately`).

<details><summary>💡 Hint 1 — what to think about</summary>

Admission is a sequence of refusals, then one of two actions. In what order should you
validate: the message, then whether a task is named, then whether it exists, then its
context, then its state? Ask of each check whether it must run before something changes
anything on the server.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Decide "is there a task id" by asking whether the key is present and not null, not whether
the value is truthy. Raise `A2AError(code, message)` for each refusal. For the answer,
validate `historyLength` before you admit anything, run `server.run_until_settled` only when
`returnImmediately` is not true, and build the reply from `server.task_view(record)`.
For the history: list slicing gives "the last n" in one expression, but what does the slice
for "the last 0" return, and is that what a client asking for none expects?
</details>

In [ ]:
def apply_history_length(task: dict, history_length: int | None) -> dict:
    """Trim `task["history"]` to the last `history_length` messages; returns the task.

    None leaves the task alone. 0 removes the `history` key entirely. A task with no history
    is returned as is. `task` is a private copy: you may modify it.

    Example:
        >>> apply_history_length({"id": "t", "history": ["a", "b", "c"]}, 2)["history"]
        ['b', 'c']
        >>> "history" in apply_history_length({"id": "t", "history": ["a"]}, 0)
        False
    """
    # YOUR CODE HERE
    raise NotImplementedError


def admit_message(server: A2AServer, params: dict) -> TaskRecord:
    """Validate the client's message and start or continue a task; return its TaskRecord.

    Raises A2AError with INVALID_PARAMS (bad message, or a contextId that does not match the
    task's), TASK_NOT_FOUND (a taskId that names no task), UNSUPPORTED_OPERATION (the task is
    terminal). A refusal must change nothing: no task created, no history appended.

    Example:
        >>> server = A2AServer(make_job)
        >>> msg = {"messageId": "m1", "role": ROLE_USER, "parts": [{"text": "solve the chain"}]}
        >>> admit_message(server, {"message": msg}).state
        'TASK_STATE_SUBMITTED'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def op_send_message(server: A2AServer, params: dict) -> dict:
    """The `SendMessage` handler. Returns `{"task": <Task>}`.

    Blocks (runs the task until it is terminal or interrupted) unless
    `params["configuration"]["returnImmediately"]` is true; trims the returned history to
    `configuration.historyLength` with `apply_history_length`. An invalid `historyLength` is
    INVALID_PARAMS and must not create a task.

    Example (a task that needs one more tick after creation):
        >>> op_send_message(server, {"message": msg, "configuration": {"returnImmediately": True}}
        ...                 )["task"]["status"]["state"]
        'TASK_STATE_SUBMITTED'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def user_message(text: str, task_id: str | None = None, context_id: str | None = None,
                 message_id: str = "m-probe") -> dict:
    """A client Message with one text part. Given."""
    message = {"messageId": message_id, "role": ROLE_USER, "parts": [{"text": text}]}
    if task_id is not None:
        message["taskId"] = task_id
    if context_id is not None:
        message["contextId"] = context_id
    return message


def _check_send_message() -> None:
    hist = ["a", "b", "c"]
    assert apply_history_length({"history": list(hist)}, None) == {"history": hist}, "None must not trim"
    assert apply_history_length({"history": list(hist)}, 2)["history"] == ["b", "c"], "n=2 keeps the LAST two"
    assert apply_history_length({"history": list(hist)}, 5)["history"] == hist, "n above the length keeps all"
    assert "history" not in apply_history_length({"history": list(hist)}, 0), (
        "historyLength 0 means NO messages: the key must be omitted (hist[-0:] is the whole list)")
    assert apply_history_length({"id": "t"}, 1) == {"id": "t"}, "a task without history must survive"

    solve = f"solve the chain from {_DEMO_TASK.start_key}"
    server = A2AServer(make_job)
    got = op_send_message(server, {"message": user_message(solve)})
    assert set(got) == {"task"}, f"SendMessage must answer with exactly one of task/message, got {sorted(got)!r}"
    task = got["task"]
    assert task["status"]["state"] == COMPLETED, (
        f"a blocking SendMessage returned {task['status']['state']}: by default the call must "
        "wait until the task is terminal or interrupted")
    assert task["contextId"] and task["id"] in server.tasks, (
        "the response must carry the server-generated task id and a contextId")
    quick = A2AServer(make_job)
    early = op_send_message(quick, {"message": user_message(solve),
                                    "configuration": {"returnImmediately": True}})["task"]
    assert early["status"]["state"] == SUBMITTED, (
        f"returnImmediately: true returned {early['status']['state']}; it must return right after "
        "the task is created, before the work is done")
    asking = op_send_message(A2AServer(make_job), {"message": user_message("solve the chain")})["task"]
    assert asking["status"]["state"] == INPUT_REQUIRED, (
        "a blocking call must also return when the task is INTERRUPTED, not only when it is terminal")

    def refused(server, params, code):
        before = {tid: len(rec.history) for tid, rec in server.tasks.items()}
        try:
            admit_message(server, params)
        except A2AError as exc:
            assert exc.code == code, f"expected code {code}, got {exc.code} ({exc.message})"
            after = {tid: len(rec.history) for tid, rec in server.tasks.items()}
            assert after == before, "a refused message left a trace: a task or a history entry changed"
            return
        raise AssertionError(f"the message was accepted; it must be refused with {code}")

    refused(server, {"message": user_message("x", task_id="no-such-task")}, TASK_NOT_FOUND)
    refused(server, {"message": user_message("x", task_id="")}, TASK_NOT_FOUND)
    refused(server, {"message": user_message("x", task_id=task["id"])}, UNSUPPORTED_OPERATION)
    refused(A2AServer(make_job), {"message": {"messageId": "m", "role": ROLE_USER, "parts": []}},
            INVALID_PARAMS)
    refused(A2AServer(make_job), {}, INVALID_PARAMS)
    fresh = A2AServer(make_job)
    open_task = op_send_message(fresh, {"message": user_message("solve the chain")})["task"]
    refused(fresh, {"message": user_message("x", task_id=open_task["id"], context_id="other")},
            INVALID_PARAMS)
    n_tasks = len(fresh.tasks)
    try:
        op_send_message(fresh, {"message": user_message(solve),
                                "configuration": {"historyLength": -1}})
    except A2AError as exc:
        assert exc.code == INVALID_PARAMS and len(fresh.tasks) == n_tasks, (
            "an invalid historyLength must be INVALID_PARAMS and must NOT have created a task: "
            "validate before you admit")
    else:
        raise AssertionError("historyLength -1 was accepted")
    print("exercise 3 looks right: admission refuses cleanly, and SendMessage blocks unless told not to")


_try("exercise 3", _check_send_message, needs=("exercise 2",))

## 7. Exercise 4: `GetTask` and `CancelTask`

`GetTask` returns the Task for an id: the call a polling client makes over and over.
`CancelTask` asks the server to cancel: for a task that is **not terminal** this lesson's server
moves it to `CANCELED` (the specification only promises an attempt,
`a2a-cancel-not-guaranteed`); for one that has already finished it must answer
`TaskNotCancelableError` (-32002) and change nothing (`a2a-cancel-not-cancelable`). An unknown id is
`TaskNotFoundError` (-32001) for both, and a missing or non-string `id` is `INVALID_PARAMS`.
`GetTask` reads `historyLength` from its params and trims with your `apply_history_length`;
`CancelTask` returns the whole task.

<details><summary>💡 Hint 1 — what to think about</summary>

For cancel, which two checks come before the state change, and does the order between them
matter? For `GetTask`, which inputs can be wrong (the id, the `historyLength`), and which
error does each one earn?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

In `op_cancel_task`, look the task up (unknown means -32001), refuse if its state is terminal
(-32002), and only then call
`server.enter_state(record, CANCELED)`, so your state machine, not this handler, has the final
say on the move. Return the task from `server.task_view`.
</details>

In [ ]:
def op_get_task(server: A2AServer, params: dict) -> dict:
    """The `GetTask` handler: `{"id": ..., "historyLength": ...}` -> the Task itself.

    INVALID_PARAMS for a missing/non-string id or a bad historyLength; TASK_NOT_FOUND for an
    id that names no task.

    Example:
        >>> op_get_task(server, {"id": task_id, "historyLength": 0})["status"]["state"]
        'TASK_STATE_COMPLETED'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def op_cancel_task(server: A2AServer, params: dict) -> dict:
    """The `CancelTask` handler: `{"id": ...}` -> the updated Task (state CANCELED).

    INVALID_PARAMS for a missing/non-string id; TASK_NOT_FOUND for an unknown id;
    TASK_NOT_CANCELABLE if the task is already terminal (nothing may change).

    Example:
        >>> op_cancel_task(server, {"id": waiting_task_id})["status"]["state"]
        'TASK_STATE_CANCELED'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_get_and_cancel() -> None:
    server = A2AServer(make_job)
    solved = op_send_message(server, {"message": user_message(f"solve the chain from {_DEMO_TASK.start_key}")})["task"]
    got = op_get_task(server, {"id": solved["id"]})
    assert got["id"] == solved["id"] and got["status"]["state"] == COMPLETED, f"GetTask returned {got!r}"
    n_all = len(got["history"])
    one = op_get_task(server, {"id": solved["id"], "historyLength": 1})
    assert len(one["history"]) == 1 and n_all > 1, "historyLength 1 must return exactly the last message"
    assert "history" not in op_get_task(server, {"id": solved["id"], "historyLength": 0}), (
        "historyLength 0 must omit the history")

    def code_of(fn, params) -> int:
        try:
            fn(server, params)
        except A2AError as exc:
            return exc.code
        return 0

    assert code_of(op_get_task, {"id": "nope"}) == TASK_NOT_FOUND, "an unknown id is TaskNotFoundError (-32001)"
    assert code_of(op_get_task, {}) == INVALID_PARAMS and code_of(op_get_task, {"id": 7}) == INVALID_PARAMS, (
        "a missing or non-string id is INVALID_PARAMS")
    assert code_of(op_get_task, {"id": solved["id"], "historyLength": -1}) == INVALID_PARAMS, (
        "a negative historyLength is INVALID_PARAMS")
    assert code_of(op_get_task, {"id": solved["id"], "historyLength": True}) == INVALID_PARAMS, (
        "a boolean is not an integer historyLength")
    assert code_of(op_cancel_task, {"id": "nope"}) == TASK_NOT_FOUND, "cancelling an unknown task is -32001"
    assert code_of(op_cancel_task, {"id": solved["id"]}) == TASK_NOT_CANCELABLE, (
        "cancelling a COMPLETED task must be TaskNotCancelableError (-32002), not a success")
    assert op_get_task(server, {"id": solved["id"]})["status"]["state"] == COMPLETED, (
        "the refused cancel changed the finished task")
    waiting = op_send_message(server, {"message": user_message("solve the chain")})["task"]
    assert waiting["status"]["state"] == INPUT_REQUIRED, (
        f"a request with no start key must stop at INPUT_REQUIRED, got {waiting['status']['state']}")
    cancelled = op_cancel_task(server, {"id": waiting["id"]})
    assert cancelled["status"]["state"] == CANCELED, (
        f"a task in INPUT_REQUIRED is not terminal, so it can be canceled; got {cancelled['status']['state']}")
    assert code_of(op_cancel_task, {"id": waiting["id"]}) == TASK_NOT_CANCELABLE, (
        "a second cancel finds a terminal task")
    print("exercise 4 looks right: history trimming, unknown ids, and cancel of live vs finished tasks")


_try("exercise 4", _check_get_and_cancel, needs=("exercise 2", "exercise 3"))

## 8. Exercise 5: `SendStreamingMessage`

The streaming twin of `SendMessage`. The specification is explicit about its shape
(`a2a-stream-shape`, `a2a-stream-close`, `a2a-stream-exactly-one`): the operation MUST
establish a stream; if a Task is returned, the stream MUST *begin with the Task object*, then
carry zero or more status and artifact update events, and MUST close once the task reaches a
terminal state. Every event holds exactly one of `task`, `message`, `statusUpdate`,
`artifactUpdate`. The server plumbing already frames each event as `data: ...` in a
`text/event-stream` reply; your handler returns the **list of events**, in the order they
were generated (a reordering is a violation too, `a2a-event-order`).

Two more rules. A server whose card says `capabilities.streaming` is false MUST answer
`UnsupportedOperationError` (-32004) (`a2a-capability-validation`), and it must do so
**before it creates a task**. And the first event is the task *as it was when the stream
began*, so take that snapshot before you run the work. Message validation and `historyLength`
work exactly as in exercise 3 (trim the snapshot).

<details><summary>💡 Hint 1 — what to think about</summary>

Three things happen in a fixed order: a capability check, an admission, then the run. Which of
them changes the server, and so must come last? And what is the difference between the task
snapshot before the run and the task after it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Read the capability from `server.agent_card()`. After admitting the message, build
`{"task": ...}` from `server.task_view(record)` first, and only then call
`server.run_until_settled(record)`, which returns the remaining events. Put the snapshot
in front of that list.
</details>

In [ ]:
def op_stream_message(server: A2AServer, params: dict) -> list[dict]:
    """The `SendStreamingMessage` handler: returns the ordered list of StreamResponse events.

    First event: `{"task": <Task as it was before any work>}`; then the status and artifact
    events the work produces. UNSUPPORTED_OPERATION, with no task created, if the Agent Card
    declares streaming false.

    Example (the shape, for a task that is completed by the end):
        >>> [next(iter(event)) for event in op_stream_message(server, {"message": msg})][:2]
        ['task', 'statusUpdate']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_stream_message() -> None:
    server = A2AServer(make_job)
    events = op_stream_message(server, {"message": user_message(f"solve the chain from {_DEMO_TASK.start_key}")})
    kinds = [next(iter(event)) for event in events]
    assert kinds[0] == "task", f"the stream must BEGIN with the Task object; it began with {kinds[0]!r}"
    assert events[0]["task"]["status"]["state"] == SUBMITTED, (
        "the first event is the task as it was when the stream began (SUBMITTED), not the finished one: "
        "take the snapshot before you run the work")
    assert all(len(event) == 1 for event in events), "every event must hold exactly one of task/statusUpdate/artifactUpdate/message"
    assert set(kinds[1:]) <= {"statusUpdate", "artifactUpdate"}, f"unexpected event kinds after the task: {kinds[1:]!r}"
    last = events[-1].get("statusUpdate", {}).get("status", {}).get("state")
    assert last == COMPLETED, f"the stream must end at the terminal state; its last event says {last!r}"
    assert sum(1 for e in events if e.get("statusUpdate", {}).get("status", {}).get("state") in TERMINAL_STATES) == 1, (
        "exactly one terminal status event, and it is the last one")
    assert "artifactUpdate" in kinds and len(events) > 3, "progress and artifact events must be in the stream"
    off = A2AServer(make_job, streaming=False)
    try:
        op_stream_message(off, {"message": user_message("solve the chain")})
    except A2AError as exc:
        assert exc.code == UNSUPPORTED_OPERATION, f"streaming is not declared: expected -32004, got {exc.code}"
        assert not off.tasks, "the capability check must come BEFORE admission: a task was created"
    else:
        raise AssertionError("a server whose card says streaming is false served a stream")
    asks = op_stream_message(A2AServer(make_job), {"message": user_message("solve the chain")})
    end = asks[-1]["statusUpdate"]["status"]["state"]
    assert end == INPUT_REQUIRED, f"this lesson's server ends a stream at an interrupted state too; ended at {end!r}"
    print("exercise 5 looks right: a real event sequence, task first, closing at the settled state")


_try("exercise 5", _check_stream_message, needs=("exercise 2", "exercise 3"))

## 9. Given: the client side, and a first end-to-end run

`Loopback` is the in-memory transport: `post(bytes)` goes to the server's `handle`, `get(path)`
to its discovery route, and it records every message in both directions (section 14 reads that
record). `sleep()` is how a client "waits": it lets the server's logical clock tick once, the
in-memory stand-in for time passing. `A2AClient` builds the JSON-RPC envelope with a fresh id,
sends the `A2A-Version` header on every request (the specification says a client MUST), checks
that the response id matches, and turns an error object into an exception. You write the parts
that need judgement: how to *follow* a task, how to *fold a stream*, and what to do when the
agent asks a question.

In [ ]:
class Loopback:
    """An in-memory transport: bytes in, bytes out, nothing else. Records every message."""

    def __init__(self, server: A2AServer):
        self.server = server
        self.transcript: list[tuple[str, bytes]] = []      # (direction, raw bytes)

    def post(self, raw: bytes, headers: dict | None = None) -> Reply:
        self.transcript.append(("client->server", raw))
        reply = self.server.handle(raw, headers)
        self.transcript.extend(("server->client", chunk) for chunk in reply.chunks)
        return reply

    def get(self, path: str) -> Reply:
        self.transcript.append(("client->server", f"GET {path}".encode()))
        reply = self.server.get(path)
        self.transcript.extend(("server->client", chunk) for chunk in reply.chunks)
        return reply

    def sleep(self) -> None:
        """Waiting, in memory: one tick of the server's logical clock. No wall clock is read."""
        self.server.advance(1)


class A2AClient:
    """A minimal A2A 1.0 client over any transport with post(), get() and sleep()."""

    def __init__(self, transport: Loopback, version: str = A2A_VERSION):
        self.transport, self.version = transport, version
        self._next_id, self._next_message = 1, 1

    def _envelope(self, method: str, params: dict) -> tuple[int, bytes]:
        rid, self._next_id = self._next_id, self._next_id + 1
        return rid, _dumps({"jsonrpc": "2.0", "id": rid, "method": method, "params": params})

    def _headers(self) -> dict:
        return {"A2A-Version": self.version}

    def call(self, method: str, params: dict) -> Any:
        """One request, one JSON answer: returns `result`, raises A2ARemoteError on an error object."""
        rid, raw = self._envelope(method, params)
        reply = self.transport.post(raw, self._headers())
        if reply.content_type != JSON_CT or len(reply.chunks) != 1:
            raise A2AWireError(f"expected one {JSON_CT} body, got {reply.content_type}")
        body = json.loads(reply.chunks[0])
        if body.get("id") != rid:
            raise A2AWireError(f"response id {body.get('id')!r} does not match request id {rid!r}")
        if "error" in body:
            raise A2ARemoteError(body["error"]["code"], body["error"]["message"])
        return body["result"]

    def stream(self, method: str, params: dict) -> Iterator[dict]:
        """One request, a stream of StreamResponse events (each unwrapped from its JSON-RPC envelope)."""
        rid, raw = self._envelope(method, params)
        reply = self.transport.post(raw, self._headers())
        if reply.content_type != SSE_CT:
            body = json.loads(reply.chunks[0]) if len(reply.chunks) == 1 else {}
            if "error" in body:
                raise A2ARemoteError(body["error"]["code"], body["error"]["message"])
            raise A2AWireError(f"expected a {SSE_CT} stream but got {reply.content_type}: "
                               "a streaming request answered with a single blob")
        for chunk in reply.chunks:
            if not (chunk.startswith(b"data: ") and chunk.endswith(b"\n\n")):
                raise A2AWireError("a stream event must be 'data: <json>' followed by a blank line")
            body = json.loads(chunk[len(b"data: "):])
            if body.get("id") != rid or "result" not in body:
                raise A2AWireError(f"stream event with a wrong id or no result: {body!r}")
            yield body["result"]

    def user_message(self, text: str, task_id: str | None = None,
                     context_id: str | None = None) -> dict:
        number, self._next_message = self._next_message, self._next_message + 1
        return user_message(text, task_id, context_id, message_id=f"client-msg-{number}")

    def send(self, text: str, task_id: str | None = None, context_id: str | None = None,
             return_immediately: bool = False) -> dict:
        """SendMessage: returns the result object (`{"task": ...}` or `{"message": ...}`)."""
        params: dict = {"message": self.user_message(text, task_id, context_id)}
        if return_immediately:
            params["configuration"] = {"returnImmediately": True}
        return self.call("SendMessage", params)

    def fetch_card(self) -> dict:
        reply = self.transport.get(WELL_KNOWN_CARD)
        if reply.status != 200:
            raise A2AWireError(f"no Agent Card at {WELL_KNOWN_CARD}: HTTP {reply.status}")
        return json.loads(reply.chunks[0])


def new_run(verbose: bool = False, streaming: bool = True) -> tuple[A2AServer, Loopback, A2AClient]:
    """A fresh server, transport and client wired together."""
    server = A2AServer(lambda: make_job(verbose=verbose), streaming=streaming)
    transport = Loopback(server)
    return server, transport, A2AClient(transport)


def _first_run() -> None:
    server, transport, client = new_run()
    card = client.fetch_card()
    problems = validate_agent_card(card)
    print(f"fetched the card of {card['name']!r}: {len(card['skills'])} skill(s), streaming "
          f"{card['capabilities']['streaming']}, {len(problems)} problem(s)")
    task = client.send(f"solve the chain from {_DEMO_TASK.start_key}")["task"]
    artifact = "".join(p["text"] for p in task["artifacts"][0]["parts"])
    print(f"SendMessage blocked, then returned a task in state {task['status']['state']}")
    print(f"  artifact {task['artifacts'][0]['name']!r}: {artifact!r} (target {_DEMO_TASK.target})")
    print(f"  history holds {len(task['history'])} messages; {len(transport.transcript)} wire messages so far")


_try("first run", _first_run, needs=("exercise 1", "exercise 2", "exercise 3"))

## 10. Exercise 6: following a long-running task by polling

A task can take a while. The client starts it with `returnImmediately: true`, gets a task in an
early state back, and then asks `GetTask` until the task has *settled*. Settled means
**terminal or interrupted**: an `INPUT_REQUIRED` (or `AUTH_REQUIRED`) task will not progress
until the client does something, so polling it further is pointless, and it is not a failure
(`a2a-input-required`). Write `poll_until_settled(call, task_id, sleep, max_polls)`:

* call `GetTask` (through `call(method, params)`, which returns the Task) at most `max_polls`
  times, the first time immediately;
* return the task the first time its state is terminal or interrupted;
* wait (`sleep()`) only *between* calls: never after the call that settled, and never after the
  last allowed poll;
* if the task is still not settled after `max_polls` calls, raise `PollTimeout`.

<details><summary>💡 Hint 1 — what to think about</summary>

Two counts are in play: how many times you asked and how many times you waited. When the very
first answer is already settled, how many of each should there be? When the last allowed poll
is the one that settles, does the result depend on whether you waited before it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

A counted loop over the polls; inside it, ask, test the state against the two settled sets,
return if it is in either, and otherwise wait only if another poll is still allowed. Raise
the timeout after the loop, not inside it.
</details>

In [ ]:
class PollTimeout(Exception):
    """The task was still running after the client's last allowed poll."""


def poll_until_settled(call: Callable[[str, dict], dict], task_id: str,
                       sleep: Callable[[], None], max_polls: int = 50) -> dict:
    """Poll `GetTask` until the task is terminal or interrupted; return that Task.

    Makes at most `max_polls` calls; `sleep()` runs between calls only. Raises PollTimeout if
    the task is still SUBMITTED or WORKING after the last poll.

    Example (a task that is WORKING for two polls, then COMPLETED, with max_polls=5):
        3 calls to GetTask, 2 calls to sleep, returns the COMPLETED task.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_poll() -> None:
    def scripted(states: list[str]):
        log = {"calls": 0, "sleeps": 0}

        def call(method, params):
            assert method == "GetTask" and params == {"id": "t1"}, f"polled with {method!r} {params!r}"
            if log["calls"] >= 100:
                raise RunawayLoop("GetTask was polled 100 times in a check that allows at most 5: stop "
                                  "after max_polls calls, and treat an INTERRUPTED state as settled")
            state = states[min(log["calls"], len(states) - 1)]
            log["calls"] += 1
            return {"id": "t1", "status": {"state": state}}

        def sleep():
            log["sleeps"] += 1
        return call, sleep, log

    call, sleep, log = scripted([WORKING, WORKING, COMPLETED])
    task = poll_until_settled(call, "t1", sleep, max_polls=5)
    assert task["status"]["state"] == COMPLETED, f"returned {task!r}"
    assert (log["calls"], log["sleeps"]) == (3, 2), (
        f"{log['calls']} polls and {log['sleeps']} waits: expected to ask until settled, waiting only "
        "between polls (never after the poll that settled)")
    call, sleep, log = scripted([COMPLETED])
    poll_until_settled(call, "t1", sleep, max_polls=5)
    assert (log["calls"], log["sleeps"]) == (1, 0), "a task settled at the first poll needs no wait at all"
    call, sleep, log = scripted([WORKING, INPUT_REQUIRED])
    task = poll_until_settled(call, "t1", sleep, max_polls=5)
    assert task["status"]["state"] == INPUT_REQUIRED and log["calls"] == 2, (
        "INPUT_REQUIRED is settled (the task waits for the client): return it, do not keep polling "
        "and do not treat it as a failure")
    call, sleep, log = scripted([SUBMITTED, WORKING, AUTH_REQUIRED])
    assert poll_until_settled(call, "t1", sleep, max_polls=3)["status"]["state"] == AUTH_REQUIRED, (
        "AUTH_REQUIRED is an interrupted state too")
    call, sleep, log = scripted([WORKING, WORKING, WORKING, COMPLETED])
    got = poll_until_settled(call, "t1", sleep, max_polls=4)
    assert got["status"]["state"] == COMPLETED and log["calls"] == 4, (
        "a task that settles exactly on the last allowed poll must be returned, not timed out")
    call, sleep, log = scripted([WORKING, WORKING, WORKING, WORKING, COMPLETED])
    try:
        poll_until_settled(call, "t1", sleep, max_polls=4)
    except PollTimeout:
        assert (log["calls"], log["sleeps"]) == (4, 3), (
            f"{log['calls']} polls, {log['sleeps']} waits: at most max_polls polls, and no wait after the last")
    else:
        raise AssertionError("a task still running after max_polls polls must raise PollTimeout")
    print("exercise 6 looks right: settles on terminal or interrupted, waits only between polls")


_try("exercise 6", _check_poll)


def _poll_demo() -> None:
    server, transport, client = new_run()
    task = client.send(f"solve the chain from {_DEMO_TASK.start_key}", return_immediately=True)["task"]
    seen = [task["status"]["state"]]
    polls = {"n": 0}

    def call(method, params):
        polls["n"] += 1
        result = client.call(method, params)
        seen.append(result["status"]["state"])
        return result

    done = poll_until_settled(call, task["id"], transport.sleep, max_polls=30)
    names = [s.replace("TASK_STATE_", "") for s in seen]
    print(f"returnImmediately gave {names[0]}; polling {polls['n']} time(s) saw: " + " > ".join(names[1:]))
    print(f"settled in {done['status']['state']} with {len(done.get('artifacts', []))} artifact(s)")


_try("polling demo", _poll_demo, needs=("exercise 2", "exercise 3", "exercise 4", "exercise 6"))

## 11. Exercise 7: folding a stream into a task view

Streaming gives the client the *events*, not the task. To know where things stand it folds
them into a view. The fold has rules of its own. A `statusUpdate` **replaces** the state and
the status message. An `artifactUpdate` with `append: true` **adds** its parts to an artifact
already received under the same `artifactId`; without `append` it **replaces** that artifact;
`lastChunk: true` marks the artifact complete (`a2a-artifact-update`). A stream that breaks
the specification must be *noticed*, so raise `A2AWireError` for: an event that does not hold
exactly one of `task`, `message`, `statusUpdate`, `artifactUpdate` (`a2a-stream-exactly-one`);
a first event that is not a `task` or `message`; a `task` or `message` anywhere but first; an
update for a different task than the first event's; and **any event after** a terminal state
or after a lone `message` (the stream MUST close there, `a2a-stream-close`).

Write `apply_stream_event(view, event)`, which updates `view` in place (start from
`new_view()`): `view["events"]` counts the events folded so far, and each artifact is
`{"name", "text", "complete"}` with `text` the concatenation of its parts' text.

<details><summary>💡 Hint 1 — what to think about</summary>

Separate *checking* the event from *applying* it. The checks all need only the event and the
view's current state. For the artifact, what must be true of the view before an `append: true`
chunk can be appended to something, and what should happen if it is not?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Find which of the four keys are present, and raise unless there is exactly one. Then apply the
ordering checks using the event counter and the current state. Then branch on the kind: each
branch sets a few fields of the view. For an artifact chunk, extend the existing text only when
`append` is true and the artifact exists; otherwise store a new entry.
</details>

In [ ]:
def new_view() -> dict:
    """An empty client-side view of a task, to fold stream events into. Given."""
    return {"taskId": None, "contextId": None, "state": None, "statusText": None,
            "artifacts": {}, "message": None, "events": 0}


def apply_stream_event(view: dict, event: dict) -> None:
    """Fold one StreamResponse event into `view`, in place; raise A2AWireError on a violation.

    Example:
        >>> view = new_view()
        >>> apply_stream_event(view, {"task": {"id": "t", "contextId": "c", "status": {"state": WORKING}}})
        >>> apply_stream_event(view, {"statusUpdate": {"taskId": "t", "contextId": "c",
        ...                                            "status": {"state": COMPLETED}}})
        >>> view["state"], view["events"]
        ('TASK_STATE_COMPLETED', 2)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def follow_stream(client: A2AClient, text: str, task_id: str | None = None,
                  context_id: str | None = None) -> dict:
    """SendStreamingMessage, folded through your apply_stream_event. Returns the final view. Given."""
    view = new_view()
    message = client.user_message(text, task_id, context_id)
    for event in client.stream("SendStreamingMessage", {"message": message}):
        apply_stream_event(view, event)
    return view


def _check_fold() -> None:
    def task(state, tid="t", artifacts=None):
        body = {"id": tid, "contextId": "c", "status": {"state": state}}
        if artifacts:
            body["artifacts"] = artifacts
        return {"task": body}

    def status(state, text=None, tid="t"):
        s = {"state": state}
        if text:
            s["message"] = {"messageId": "m", "role": ROLE_AGENT, "parts": [{"text": text}]}
        return {"statusUpdate": {"taskId": tid, "contextId": "c", "status": s}}

    def chunk(aid, text, append=False, last=False, tid="t"):
        return {"artifactUpdate": {"taskId": tid, "contextId": "c", "append": append, "lastChunk": last,
                                   "artifact": {"artifactId": aid, "name": "n", "parts": [{"text": text}]}}}

    def fold(*events):
        view = new_view()
        for event in events:
            apply_stream_event(view, event)
        return view

    def breaks(*events) -> bool:
        try:
            fold(*events)
        except A2AWireError:
            return True
        return False

    v = fold(task(SUBMITTED), status(WORKING, "started"), chunk("a", "The chain ends at "),
             chunk("a", "623", append=True, last=True), status(COMPLETED))
    assert v["state"] == COMPLETED and v["taskId"] == "t" and v["contextId"] == "c" and v["events"] == 5, f"view {v!r}"
    assert v["artifacts"]["a"] == {"name": "n", "text": "The chain ends at 623", "complete": True}, (
        f"artifact folded to {v['artifacts']['a']!r}: append: true adds to the artifact received earlier, "
        "lastChunk marks it complete")
    mid = fold(task(SUBMITTED), status(WORKING, "step"), chunk("a", "one"))
    assert mid["statusText"] == "step" and mid["artifacts"]["a"]["complete"] is False, (
        "a statusUpdate carries the new status message; an artifact is not complete before its lastChunk")
    assert fold(task(WORKING), status(WORKING, "step"), status(INPUT_REQUIRED))["statusText"] is None, (
        "a statusUpdate REPLACES the status: with no message, the old status text must be gone")
    assert fold(task(WORKING), chunk("a", "old"), chunk("a", "new"))["artifacts"]["a"]["text"] == "new", (
        "an artifactUpdate WITHOUT append replaces the artifact")
    assert fold(task(WORKING), chunk("a", "solo", append=True))["artifacts"]["a"]["text"] == "solo", (
        "an append chunk with nothing to append to becomes the artifact instead of crashing")
    seeded = fold(task(WORKING, artifacts=[{"artifactId": "a", "name": "n", "parts": [{"text": "x"}, {"text": "y"}]}]))
    assert seeded["artifacts"]["a"]["text"] == "xy", "the artifacts inside a task event are folded in, parts joined"
    assert fold({"message": {"messageId": "m", "role": ROLE_AGENT, "parts": [{"text": "hi"}]}})["message"] == "hi", (
        "a message-only stream folds to the message text")
    assert breaks({"task": {"id": "t", "status": {"state": WORKING}}, "statusUpdate": {}}), (
        "an event with two payloads is not a valid StreamResponse")
    assert breaks({}), "an empty event is not a valid StreamResponse"
    assert breaks(status(WORKING)), "a stream must BEGIN with the task"
    assert breaks(task(WORKING), task(WORKING)), "a second task event in one stream is a violation"
    for done in (COMPLETED, FAILED, CANCELED, REJECTED):
        assert breaks(task(WORKING), status(done), status(WORKING)), (
            f"an event after {done} must be flagged: the stream MUST close at a terminal state")
        assert breaks(task(WORKING), status(done), chunk("a", "late")), (
            f"an ARTIFACT after {done} must be flagged too (an out-of-order stream)")
    assert not breaks(task(WORKING), status(INPUT_REQUIRED)), "an interrupted state is a normal thing to fold"
    assert breaks(task(WORKING, tid="t"), status(WORKING, tid="other")), "an update for another task is a violation"
    assert breaks({"message": {"messageId": "m", "role": ROLE_AGENT, "parts": [{"text": "hi"}]}}, status(WORKING)), (
        "nothing may follow a message-only stream's single message")
    print("exercise 7 looks right: append vs replace, lastChunk, and every stream violation noticed")


_try("exercise 7", _check_fold)


def _stream_demo() -> None:
    server, transport, client = new_run()
    view = follow_stream(client, f"solve the chain from {_DEMO_TASK.start_key}")
    art = next(iter(view["artifacts"].values()))
    print(f"streamed {view['events']} events; final state {view['state']}")
    print(f"  folded artifact {art['name']!r} = {art['text']!r}, complete: {art['complete']}")
    _, transport2, client2 = new_run(streaming=False)
    try:
        follow_stream(client2, "solve the chain")
    except A2ARemoteError as exc:
        print(f"a server that declares no streaming answers: {exc}")


_try("streaming demo", _stream_demo, needs=("exercise 2", "exercise 3", "exercise 5", "exercise 7"))

## 12. Exercise 8: answering `input-required`

When the agent needs something from the client it moves the task to `INPUT_REQUIRED` and puts
its question in the status message. That is **not an error and not the end**: the task is alive,
and the client continues it by sending a new message **with the same `taskId` and `contextId`**
(`a2a-input-required`). Send it without the `taskId` and you have started a *new* task.

Write `converse(send, first_text, answer, max_rounds)`. `send(text, task_id, context_id)`
returns the `SendMessage` result (`{"task": ...}` or `{"message": ...}`); `answer(prompt)`
turns the agent's question into the client's reply text.

* `COMPLETED`: return `Conversation(task, None, rounds, prompts)`; a direct `message` result is
  an answer too: `Conversation(None, its text, rounds, prompts)`;
* `INPUT_REQUIRED`: pass the status message's text to `answer`, then `send` the reply with the
  task's own `id` and `contextId`; at most `max_rounds` such answers, then `ConversationError`;
* `SUBMITTED` or `WORKING` (an unsettled task): `ConversationError`, it needs polling instead;
* any other state (`FAILED`, `REJECTED`, `CANCELED`, `AUTH_REQUIRED`): `TaskNotCompleted(state, text)`.

<details><summary>💡 Hint 1 — what to think about</summary>

A loop whose body is "look at the state of what came back". Which of the states means "go
round again"? What must the follow-up carry that the first message did not, and where do you
get those values from?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Handle the no-task (`message`) result first. Then decide by state: done, ask again, or refuse.
Count only the answers you actually send, and check the cap *before* answering, so exactly
`max_rounds` answers are allowed and the next question raises.
</details>

In [ ]:
class Conversation(NamedTuple):
    task: dict | None
    message: str | None
    rounds: int
    prompts: tuple[str, ...]


class ConversationError(Exception):
    """The exchange could not finish: too many questions, or an unsettled task."""


class TaskNotCompleted(Exception):
    """The remote agent ended the task without completing it (or needs authentication)."""

    def __init__(self, state: str, text: str):
        super().__init__(f"the task ended in {state}: {text!r}")
        self.state, self.text = state, text


def converse(send: Callable[[str, str | None, str | None], dict], first_text: str,
             answer: Callable[[str], str], max_rounds: int = 3) -> Conversation:
    """Send `first_text`, answer the agent's questions, and return the completed outcome.

    Example (agent asks one question, then completes):
        >>> converse(send, "solve the chain", lambda q: "start from k0_012345").rounds
        1
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_converse() -> None:
    def task(state, text=None, tid="t9", ctx="c9"):
        status = {"state": state}
        if text:
            status["message"] = {"messageId": "m", "role": ROLE_AGENT, "parts": [{"text": text}]}
        return {"task": {"id": tid, "contextId": ctx, "status": status}}

    def scripted(*results):
        sent = []

        def send(text, task_id, context_id):
            sent.append((text, task_id, context_id))
            return results[len(sent) - 1]
        return send, sent

    send, sent = scripted(task(INPUT_REQUIRED, "which key?"), task(COMPLETED))
    out = converse(send, "solve it", lambda q: f"answer to {q}")
    assert out.task["status"]["state"] == COMPLETED and out.rounds == 1 and out.prompts == ("which key?",), (
        f"got {out!r}: an INPUT_REQUIRED task is a question to answer, not a failure")
    assert sent == [("solve it", None, None), ("answer to which key?", "t9", "c9")], (
        f"the follow-up must carry the task's own id AND contextId; the calls were {sent!r}: "
        "without the taskId the server starts a NEW task")
    send, sent = scripted(task(COMPLETED))
    assert converse(send, "x", lambda q: "unused").rounds == 0 and len(sent) == 1, "no question, no answer"
    send, sent = scripted(*([task(INPUT_REQUIRED, "again?")] * 2 + [task(COMPLETED)]))
    assert converse(send, "x", lambda q: "a", max_rounds=2).rounds == 2, (
        "exactly max_rounds answers are allowed: two questions with max_rounds=2 must complete")
    send, sent = scripted(*([task(INPUT_REQUIRED, "again?")] * 3 + [task(COMPLETED)]))
    try:
        converse(send, "x", lambda q: "a", max_rounds=2)
    except ConversationError:
        assert len(sent) == 3, f"{len(sent)} sends: check the cap before answering, not after"
    else:
        raise AssertionError("a third question with max_rounds=2 must raise ConversationError")
    for state in (FAILED, REJECTED, CANCELED, AUTH_REQUIRED):
        send, _ = scripted(task(state, "no"))
        try:
            converse(send, "x", lambda q: "a")
        except TaskNotCompleted as exc:
            assert exc.state == state and exc.text == "no", f"TaskNotCompleted carried {exc.state!r}, {exc.text!r}"
        else:
            raise AssertionError(f"a task that ended in {state} must raise TaskNotCompleted")
    for state in (SUBMITTED, WORKING):
        send, _ = scripted(task(state))
        try:
            converse(send, "x", lambda q: "a")
        except ConversationError:
            pass
        else:
            raise AssertionError(f"an unsettled {state} task must raise ConversationError")
    reply = {"message": {"messageId": "m", "role": ROLE_AGENT, "parts": [{"text": "hello back"}]}}
    send, _ = scripted(reply)
    out = converse(send, "hi", lambda q: "a")
    assert out.task is None and out.message == "hello back", f"a direct Message reply is an answer: {out!r}"
    print("exercise 8 looks right: questions are answered on the same task, failures are named")


_try("exercise 8", _check_converse)


def _converse_demo() -> None:
    server, transport, client = new_run()
    seen = []

    def answer(prompt: str) -> str:
        seen.append(prompt)
        return f"start from {_DEMO_TASK.start_key}"

    out = converse(lambda t, tid, cid: client.send(t, tid, cid), "please solve the lookup chain", answer)
    art = out.task["artifacts"][0]
    print(f"the agent asked: {seen[0]!r}")
    print(f"answered on the same task; completed after {out.rounds} question(s); tasks on the server: {len(server.tasks)}")
    print(f"  artifact text: {''.join(p['text'] for p in art['parts'])!r}")
    server2, _, client2 = new_run()
    try:
        converse(lambda t, tid, cid: client2.send(t, tid, cid), "hello there", answer)
    except TaskNotCompleted as exc:
        print(f"a request the agent will not do: {exc}")


_try("conversation demo", _converse_demo,
     needs=("exercise 2", "exercise 3", "exercise 8"))

## 13. A conformance harness from the specification's own MUSTs

You have now written a server and a client, and each passed *your reading* of the rules. A
conformance harness removes the reader: it is a list of **rules**, each quoting one normative
statement from the specification (the ids in `claims.yaml`), each a probe that sends real
messages to a server and returns `None` if the server behaves or a sentence saying what it did
instead. This is the method T07-L02 used for MCP. The harness is the only part of A2A that a
third party can run against *your* server without reading your code, so a rule that cannot
fail is worthless: after you write yours, the lesson runs the whole harness against your server
**and against servers with one planted fault each**, and counts what each fault trips.


First, the harness's vocabulary and the rules you are given. Then the planted faults. Then the
four rules you write.

In [ ]:
_PROBES = {"n": 0}


def probe_raw(server: A2AServer, method: str, params: dict | None = None,
              headers: dict | None = None) -> Reply:
    """POST one JSON-RPC request to `server` and return the raw Reply. `headers=None` sends the
    correct A2A-Version; pass `{}` to send none. Given."""
    _PROBES["n"] += 1
    raw = _dumps({"jsonrpc": "2.0", "id": f"probe-{_PROBES['n']}", "method": method,
                  "params": params if params is not None else {}})
    return server.handle(raw, {"A2A-Version": A2A_VERSION} if headers is None else headers)


def probe_call(server: A2AServer, method: str, params: dict | None = None,
               headers: dict | None = None) -> dict:
    """One JSON request; returns the parsed JSON-RPC body (with `result` or `error`). Given."""
    return json.loads(probe_raw(server, method, params, headers).chunks[0])


def probe_send(server: A2AServer, text: str, task_id: str | None = None,
               context_id: str | None = None, config: dict | None = None) -> dict:
    """SendMessage with one text message; returns the parsed JSON-RPC body. Given."""
    params: dict = {"message": user_message(text, task_id, context_id)}
    if config:
        params["configuration"] = config
    return probe_call(server, "SendMessage", params)


def result_task(body: dict) -> dict:
    """The Task inside a SendMessage answer, or AssertionError saying what came back instead. Given."""
    if "result" not in body or "task" not in body["result"]:
        raise AssertionError(f"expected a task result, got {body!r}")
    return body["result"]["task"]


def reach_state(server: A2AServer, state: str) -> str:
    """Drive `server` until it holds a task in `state`; return that task's id. Given.

    COMPLETED, FAILED, REJECTED: one blocking SendMessage. INPUT_REQUIRED: a request with no
    start key. CANCELED: that request, then CancelTask. (Cancelling is not guaranteed by the
    specification, so against a server that never cancels, CANCELED cannot be reached and a rule
    that needs a CANCELED task cannot run: a limit of the probe, not a verdict on the server.)"""
    text = {COMPLETED: f"solve the chain from {_DEMO_TASK.start_key}",
            FAILED: "solve the chain from k9_000000", REJECTED: "hello there",
            INPUT_REQUIRED: "solve the chain", CANCELED: "solve the chain"}[state]
    task = result_task(probe_send(server, text))
    if state == CANCELED:
        task = probe_call(server, "CancelTask", {"id": task["id"]})["result"]
    if task["status"]["state"] != state:
        raise AssertionError(f"could not bring the server to {state}: it reached {task['status']['state']!r}")
    return task["id"]


def _error_code(body: dict) -> int | None:
    return body.get("error", {}).get("code")


def _state_of(server: A2AServer, task_id: str) -> str:
    return probe_call(server, "GetTask", {"id": task_id})["result"]["status"]["state"]


# The rules the lesson gives you. Each returns None (conforms) or a sentence (what it did wrong).
def rule_card_available_and_valid(make_server) -> str | None:
    reply = make_server().get(WELL_KNOWN_CARD)
    if reply.status != 200:
        return (f"GET {WELL_KNOWN_CARD} answered {reply.status}: a server MUST make an Agent Card "
                "available, and the well-known URI is the one discovery path this harness can probe")
    try:
        card = json.loads(reply.chunks[0])
    except ValueError:
        return "the Agent Card is not valid JSON"
    problems = validate_agent_card(card)
    return ("the Agent Card is invalid: " + "; ".join(problems[:3])) if problems else None


def rule_unknown_method(make_server) -> str | None:
    body = probe_call(make_server(), "NoSuchMethod")
    return None if _error_code(body) == METHOD_NOT_FOUND else f"an unknown method must be -32601, got {body!r}"


def rule_unknown_task(make_server) -> str | None:
    server = make_server()
    for method, params in (("GetTask", {"id": "no-such-task"}), ("CancelTask", {"id": "no-such-task"})):
        code = _error_code(probe_call(server, method, params))
        if code != TASK_NOT_FOUND:
            return f"{method} on an unknown task answered {code}, expected TaskNotFoundError (-32001)"
    code = _error_code(probe_send(server, "hello", task_id="no-such-task"))
    if code != TASK_NOT_FOUND:
        return f"a message naming an unknown taskId answered {code}, expected TaskNotFoundError (-32001)"
    return None


def rule_version_negotiation(make_server) -> str | None:
    server = make_server()
    for label, headers in (("no A2A-Version header (which means 0.3)", {}),
                           ("A2A-Version: 0.3", {"A2A-Version": "0.3"}),
                           ("A2A-Version: 2.0", {"A2A-Version": "2.0"})):
        code = _error_code(probe_call(server, "GetTask", {"id": "x"}, headers))
        if code != VERSION_NOT_SUPPORTED:
            return f"{label} answered {code}, expected VersionNotSupportedError (-32009)"
    for label, headers in (("A2A-Version: 1.0", {"A2A-Version": "1.0"}),
                           ("A2A-Version: 1.0.7 (a patch number must not matter)", {"A2A-Version": "1.0.7"})):
        code = _error_code(probe_call(server, "GetTask", {"id": "x"}, headers))
        if code == VERSION_NOT_SUPPORTED:
            return f"{label} was refused with -32009; it names a supported Major.Minor"
    return None


def rule_send_blocks_until_settled(make_server) -> str | None:
    server = make_server()
    for text in (f"solve the chain from {_DEMO_TASK.start_key}", "solve the chain"):
        state = result_task(probe_send(server, text))["status"]["state"]
        if state not in TERMINAL_STATES and state not in INTERRUPTED_STATES:
            return f"the default SendMessage returned a task in {state}; it MUST wait until terminal or interrupted"
    return None


def rule_return_immediately_returns_early(make_server) -> str | None:
    server = make_server()
    task = result_task(probe_send(server, f"solve the chain from {_DEMO_TASK.start_key}",
                                  config={"returnImmediately": True}))
    state = task["status"]["state"]
    if state in TERMINAL_STATES or state in INTERRUPTED_STATES:
        return f"returnImmediately: true returned a task already in {state}; it MUST return before the work is done"
    server.advance(50)
    later = _state_of(server, task["id"])
    return None if later == COMPLETED else f"the task never finished after the early return (state {later})"


def rule_task_and_context_ids(make_server) -> str | None:
    server = make_server()
    solve = f"solve the chain from {_DEMO_TASK.start_key}"
    first, second = result_task(probe_send(server, solve)), result_task(probe_send(server, solve))
    if not first.get("id") or first["id"] == second.get("id"):
        return "two new tasks got the same id: the server MUST generate a unique taskId for each new task"
    if not first.get("contextId"):
        return "the response omits the contextId the server generated: it MUST be included"
    body = probe_send(server, solve, context_id="ctx-from-client")
    if "result" in body and result_task(body).get("contextId") != "ctx-from-client":
        return "a client contextId was silently replaced: preserve it, or reject the request"
    return None


def rule_history_length_respected(make_server) -> str | None:
    server = make_server()
    task_id = reach_state(server, COMPLETED)
    for n in (1, 0):
        got = probe_call(server, "GetTask", {"id": task_id, "historyLength": n})["result"]
        if len(got.get("history", [])) > n:
            return f"historyLength {n} returned {len(got['history'])} messages: the server MUST NOT return more"
    return None


def rule_state_names_are_protojson(make_server) -> str | None:
    server = make_server()
    for text in (f"solve the chain from {_DEMO_TASK.start_key}", "solve the chain", "hello there"):
        state = result_task(probe_send(server, text))["status"]["state"]
        if state not in ALL_STATES:
            return f"state {state!r} is not a ProtoJSON enum name such as TASK_STATE_COMPLETED (the 0.3 spelling?)"
    return None


def rule_streaming_capability_honoured(make_server) -> str | None:
    server = make_server(streaming=False)
    card = json.loads(server.get(WELL_KNOWN_CARD).chunks[0])
    if card.get("capabilities", {}).get("streaming"):
        return "this probe needs a card that declares streaming false"
    body = probe_call(server, "SendStreamingMessage", {"message": user_message("solve the chain")})
    if _error_code(body) != UNSUPPORTED_OPERATION:
        return "the card declares streaming false but SendStreamingMessage was not refused with -32004"
    return None


def rule_parts_have_exactly_one_field(make_server) -> str | None:
    task = result_task(probe_send(make_server(), f"solve the chain from {_DEMO_TASK.start_key}"))
    parts = [p for a in task.get("artifacts", []) for p in a["parts"]]
    parts += [p for m in task.get("history", []) for p in m["parts"]]
    if not parts:
        return "the solved task has no parts to inspect"
    for part in parts:
        present = [k for k in ("text", "raw", "url", "data") if k in part]
        if len(present) != 1:
            return f"a Part carries {present}: it MUST contain exactly one of text, raw, url, data"
    return None


def rule_context_mismatch_rejected(make_server) -> str | None:
    server = make_server()
    task_id = reach_state(server, INPUT_REQUIRED)
    body = probe_send(server, f"start from {_DEMO_TASK.start_key}", task_id=task_id,
                      context_id="a-different-context")
    if "error" not in body:
        return "a taskId with a mismatching contextId was accepted; the server MUST reject it"
    state = _state_of(server, task_id)
    return None if state == INPUT_REQUIRED else f"the rejected message still moved the task to {state}"

### The planted faults

Each server below subclasses the real one and breaks **one thing**, the way real
implementations do: they reopen a finished task, let a cancel succeed on one, answer a stream
with one JSON blob, speak the 0.3 spellings, ignore `historyLength`, and so on. Each fault
names the rule it was planted for. They are given, and worth reading: the difference between
the good server and each is a few lines.

In [ ]:
def _patched(table: dict, name: str, wrapper: Callable[[Callable], Callable]) -> dict:
    table[name] = wrapper(table[name])
    return table


class FaultCardMissingSkills(A2AServer):
    """The Agent Card leaves out `skills`."""
    def agent_card(self):
        card = super().agent_card()
        del card["skills"]
        return card


class FaultIgnoresVersion(A2AServer):
    """Never looks at A2A-Version."""
    def check_version(self, headers):
        return None


class FaultRevivesTerminal(A2AServer):
    """A new message quietly reopens a finished task."""
    def methods(self):
        def wrap(inner):
            def send(server, params):
                rec = server.tasks.get((params.get("message") or {}).get("taskId"))
                if rec is not None and rec.state in TERMINAL_STATES:
                    rec.state = INPUT_REQUIRED
                return inner(server, params)
            return send
        return _patched(super().methods(), "SendMessage", wrap)


class FaultCancelsFinished(A2AServer):
    """CancelTask succeeds on a task that already finished."""
    def methods(self):
        def wrap(inner):
            def cancel(server, params):
                rec = server.tasks.get(params.get("id"))
                if rec is not None and rec.state in TERMINAL_STATES:
                    rec.state = WORKING
                return inner(server, params)
            return cancel
        return _patched(super().methods(), "CancelTask", wrap)


class FaultStreamBlob(A2AServer):
    """Answers SendStreamingMessage with a single JSON body, not a stream."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stream_methods = set()

    def methods(self):
        table = super().methods()
        table["SendStreamingMessage"] = table["SendMessage"]
        return table


class FaultStreamReordered(A2AServer):
    """Streams the terminal status BEFORE the last artifact chunk."""
    def methods(self):
        def wrap(inner):
            def stream(server, params):
                events = inner(server, params)
                final = [e for e in events if e.get("statusUpdate", {}).get("status", {}).get("state") in TERMINAL_STATES]
                rest = [e for e in events if e not in final]
                arts = [i for i, e in enumerate(rest) if "artifactUpdate" in e]
                cut = arts[-1] if arts else len(rest)
                return rest[:cut] + final + rest[cut:]
            return stream
        return _patched(super().methods(), "SendStreamingMessage", wrap)


class FaultFollowUpNewTask(A2AServer):
    """Ignores the taskId in a follow-up: the answer to a question starts a new task."""
    def methods(self):
        def wrap(inner):
            def send(server, params):
                message = dict(params.get("message") or {})
                message.pop("taskId", None)
                return inner(server, dict(params, message=message))
            return send
        return _patched(super().methods(), "SendMessage", wrap)


class FaultAlwaysBlocks(A2AServer):
    """Ignores returnImmediately: it always waits."""
    def methods(self):
        def wrap(inner):
            def send(server, params):
                config = {k: v for k, v in (params.get("configuration") or {}).items() if k != "returnImmediately"}
                return inner(server, dict(params, configuration=config))
            return send
        return _patched(super().methods(), "SendMessage", wrap)


class FaultNeverBlocks(A2AServer):
    """Always returns immediately, even when the caller asked to wait (the default)."""
    def methods(self):
        def wrap(inner):
            def send(server, params):
                config = dict(params.get("configuration") or {}, returnImmediately=True)
                return inner(server, dict(params, configuration=config))
            return send
        return _patched(super().methods(), "SendMessage", wrap)


class FaultUnknownTaskIsInternalError(A2AServer):
    """An unknown task id is reported as an internal error (-32603), not -32001."""
    def methods(self):
        def wrap(inner):
            def guarded(server, params):
                try:
                    return inner(server, params)
                except A2AError as exc:
                    if exc.code == TASK_NOT_FOUND:
                        raise A2AError(INTERNAL_ERROR, "Internal error") from None
                    raise
            return guarded
        table = super().methods()
        for name in ("GetTask", "CancelTask", "SendMessage"):
            _patched(table, name, wrap)
        return table


class FaultHistoryLengthIgnored(A2AServer):
    """GetTask always returns the whole history."""
    def methods(self):
        def wrap(inner):
            return lambda server, params: inner(server, {k: v for k, v in params.items() if k != "historyLength"})
        return _patched(super().methods(), "GetTask", wrap)


class FaultTaskIdReused(A2AServer):
    """Every task gets the same id."""
    def new_task(self, message):
        rec = super().new_task(message)
        del self.tasks[rec.id]
        rec.id = "task-1"
        self.tasks[rec.id] = rec
        return rec


class FaultOldDialectStates(A2AServer):
    """Speaks the 0.3 spelling of states: `completed`, `input-required`."""
    def handle(self, raw, headers=None):
        reply = super().handle(raw, headers)
        chunks = tuple(re.sub(rb'"TASK_STATE_([A-Z_]+)"',
                              lambda m: b'"' + m.group(1).lower().replace(b"_", b"-") + b'"', c)
                       for c in reply.chunks)
        return reply._replace(chunks=chunks)


class FaultStreamsWithoutCapability(A2AServer):
    """The card says streaming false, but SendStreamingMessage is served anyway."""
    def methods(self):
        def wrap(inner):
            def stream(server, params):
                was, server.streaming = server.streaming, True
                try:
                    return inner(server, params)
                finally:
                    server.streaming = was
            return stream
        return _patched(super().methods(), "SendStreamingMessage", wrap)


class FaultTwoContentFields(A2AServer):
    """Artifact parts carry both `text` and `data`."""
    def task_view(self, rec):
        task = super().task_view(rec)
        for artifact in task.get("artifacts", []):
            for part in artifact["parts"]:
                part["data"] = {"chars": len(part.get("text", ""))}
        return task


class FaultContextMismatchAccepted(A2AServer):
    """Drops the contextId of a follow-up instead of comparing it with the task's."""
    def methods(self):
        def wrap(inner):
            def send(server, params):
                message = dict(params.get("message") or {})
                if message.get("taskId") is not None:
                    message.pop("contextId", None)
                return inner(server, dict(params, message=message))
            return send
        return _patched(super().methods(), "SendMessage", wrap)


# (class, the rule it was planted for)
FAULTS: list[tuple[type, str]] = [
    (FaultCardMissingSkills, "R01"), (FaultUnknownTaskIsInternalError, "R03"),
    (FaultIgnoresVersion, "R04"), (FaultNeverBlocks, "R05"), (FaultAlwaysBlocks, "R06"),
    (FaultTaskIdReused, "R07"), (FaultHistoryLengthIgnored, "R08"), (FaultOldDialectStates, "R09"),
    (FaultStreamsWithoutCapability, "R10"), (FaultTwoContentFields, "R11"),
    (FaultContextMismatchAccepted, "R12"), (FaultRevivesTerminal, "R13"),
    (FaultCancelsFinished, "R14"), (FaultStreamBlob, "R15"), (FaultStreamReordered, "R15"),
    (FaultFollowUpNewTask, "R16"),
]

### Exercise 9: four rules of your own

A rule takes `make_server` (a function returning a
**fresh** server; `make_server(streaming=False)` gives one without streaming) and uses the
helpers `probe_send`, `probe_call`, `probe_raw` (raw `Reply`) and `reach_state(server, state)`
(drive a fresh server until it holds a task in that state; returns the task id). Write:

* `rule_terminal_is_final`: for **each** terminal state (`COMPLETED`, `FAILED`, `REJECTED`,
  `CANCELED`), a message sent to that task must be refused with `UNSUPPORTED_OPERATION`, and
  the task must still be in the same state afterwards;
* `rule_cancel_finished_refused`: `CancelTask` on a task in `COMPLETED`, `FAILED` or
  `REJECTED` must be `TASK_NOT_CANCELABLE` with the state unchanged. Test nothing about a
  *live* task: the specification says success "is not guaranteed" (`a2a-cancel-not-guaranteed`),
  so a server that never cancels is legal and your rule must pass it;
* `rule_stream_is_a_stream`: a `SendStreamingMessage` reply must be `text/event-stream`, one
  `data: ` event per chunk, each a JSON-RPC response with `result` (the same `id` on every event),
  each result holding exactly one payload; the first is a `task` that is not yet terminal (for a
  multi-step task, the "immediate feedback on progress" of `a2a-stream-progress-feedback`), the
  rest are `statusUpdate` or `artifactUpdate`, and the only terminal status is the **last** event;
* `rule_input_required_continues`: a task in `INPUT_REQUIRED` answered with its own `taskId` and
  `contextId` must come back as the **same** task and context, and reach `COMPLETED`.

<details><summary>💡 Hint 1 — what to think about</summary>

A rule returns text, so write each check as "if the server did the wrong thing, return a
sentence naming the state or event that showed it". What is the difference between a probe
that *sees* the right behaviour once and one that would catch a server which is wrong for only
one of several states? Loop over the states, with a fresh server for each.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Each probe answers three questions in order: did it get an answer of the right *kind* (an
error, a stream, a task), was the *content* right (code, event order, ids), and did the
server's state *stay* right afterwards (read it back with a `GetTask` through `probe_call`)?
Return at the first violation; return `None` only after every state and every event has passed.
</details>

In [ ]:
# Exercise 9: the four rules you write. `make_server` returns a FRESH server each call.
def rule_terminal_is_final(make_server) -> str | None:
    """A terminal task must refuse further messages: for each of COMPLETED, FAILED, REJECTED and
    CANCELED, on a fresh server reach that state, send "one more thing" to the task, demand an
    error with code UNSUPPORTED_OPERATION, and demand the task is still in the same state.

    Returns None if the server conforms, else one sentence naming the state and what happened.

    Example (a conforming server):
        >>> rule_terminal_is_final(lambda **kw: A2AServer(make_job, **kw)) is None
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def rule_cancel_finished_refused(make_server) -> str | None:
    """CancelTask on a finished task is TaskNotCancelableError, and changes nothing. For each of
    COMPLETED, FAILED and REJECTED (fresh server each): the answer must be an error with code
    TASK_NOT_CANCELABLE and the state must be unchanged. Nothing else: a server may legally refuse
    to cancel a live task too, so a rule that demands it would fail a conforming server.

    Returns None if the server conforms, else one sentence naming the state and what happened.

    Example (a conforming server):
        >>> rule_cancel_finished_refused(lambda **kw: A2AServer(make_job, **kw)) is None
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def rule_stream_is_a_stream(make_server) -> str | None:
    """A streaming request must be answered with a stream, not a blob. Send
    SendStreamingMessage("solve the chain from <start key>") with `probe_raw`, then check the
    Reply: content type SSE_CT; every chunk is `data: <json>` + a blank line; every event is a
    JSON-RPC response with a `result`, all with the same `id`; each result holds exactly one
    payload; the first is a `task` whose state is not terminal; the others are `statusUpdate`
    or `artifactUpdate`; the only terminal `statusUpdate` is the last event.

    Returns None if the server conforms, else one sentence naming the violation.

    Example (a conforming server):
        >>> rule_stream_is_a_stream(lambda **kw: A2AServer(make_job, **kw)) is None
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def rule_input_required_continues(make_server) -> str | None:
    """Answering INPUT_REQUIRED continues the SAME task. On a fresh server send "solve the
    chain" (the agent asks for a start key), then answer with
    f"start from {_DEMO_TASK.start_key}" using the task's own id and contextId. The answer must
    come back as a task with the same `id` and `contextId`, in state COMPLETED.

    Returns None if the server conforms, else one sentence naming the violation.

    Example (a conforming server):
        >>> rule_input_required_continues(lambda **kw: A2AServer(make_job, **kw)) is None
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


class Rule(NamedTuple):
    id: str
    title: str
    claim: str          # the claims.yaml entry whose quotation this rule tests
    check: Callable


def build_rules() -> list[Rule]:
    """The whole harness: the rules you are given and the four you write. Built when called, so it
    picks up the functions as they stand now."""
    return [
        Rule("R01", "Agent Card available and valid", "a2a-card-must-be-available", rule_card_available_and_valid),
        Rule("R02", "unknown method is -32601", "a2a-jsonrpc-error-codes", rule_unknown_method),
        Rule("R03", "unknown task is -32001", "a2a-task-id-rules", rule_unknown_task),
        Rule("R04", "version negotiation", "a2a-version-negotiation", rule_version_negotiation),
        Rule("R05", "SendMessage blocks by default", "a2a-send-blocking", rule_send_blocks_until_settled),
        Rule("R06", "returnImmediately returns early", "a2a-send-return-immediately", rule_return_immediately_returns_early),
        Rule("R07", "unique task ids, contextId returned", "a2a-task-id-rules", rule_task_and_context_ids),
        Rule("R08", "historyLength respected", "a2a-history-length", rule_history_length_respected),
        Rule("R09", "states are ProtoJSON names", "a2a-protojson-enums", rule_state_names_are_protojson),
        Rule("R10", "streaming capability honoured", "a2a-capability-validation", rule_streaming_capability_honoured),
        Rule("R11", "a Part has exactly one content field", "a2a-part-exactly-one", rule_parts_have_exactly_one_field),
        Rule("R12", "contextId/taskId mismatch rejected", "a2a-context-mismatch", rule_context_mismatch_rejected),
        Rule("R13", "a terminal task is final", "a2a-terminal-rejects-messages", rule_terminal_is_final),
        Rule("R14", "cancel of a finished task refused", "a2a-cancel-not-cancelable", rule_cancel_finished_refused),
        Rule("R15", "a stream is a stream", "a2a-stream-shape", rule_stream_is_a_stream),
        Rule("R16", "input-required continues the same task", "a2a-input-required", rule_input_required_continues),
    ]


def run_rules(make_server, rules: list[Rule] | None = None) -> dict[str, str | None]:
    """Run every rule against a server factory; returns rule id -> None or the failure sentence.
    An exception inside a probe is a failure of that rule (the server broke the probe), except
    NotImplementedError, which means one of YOUR functions is not written yet, and RunawayLoop,
    which means a loop in a rule never ends: neither is a verdict about the server."""
    outcome: dict[str, str | None] = {}
    for rule in (rules if rules is not None else build_rules()):
        try:
            outcome[rule.id] = rule.check(make_server)
        except (NotImplementedError, RunawayLoop):
            raise
        except AssertionError as exc:
            outcome[rule.id] = str(exc)
        except Exception as exc:
            outcome[rule.id] = f"the probe raised {type(exc).__name__}: {exc}"
    return outcome


def demo_server_factory(cls=A2AServer):
    """A make_server function for a server class: `make_server(streaming=False)` works. Given."""
    return lambda **options: cls(make_job, **options)


def _check_rules() -> None:
    good = demo_server_factory()
    for name in _EXERCISES["exercise 9"]:
        verdict = globals()[name](good)
        assert verdict is None, (
            f"{name} rejected a server that behaves: {verdict!r}. A rule that fails a conforming "
            "server is as wrong as one that passes a broken one")
    catches = {
        "rule_terminal_is_final": FaultRevivesTerminal,
        "rule_cancel_finished_refused": FaultCancelsFinished,
        "rule_stream_is_a_stream": FaultStreamBlob,
        "rule_input_required_continues": FaultFollowUpNewTask,
    }
    for name, fault in catches.items():
        verdict = globals()[name](demo_server_factory(fault))
        assert isinstance(verdict, str) and verdict, (
            f"{name} passed {fault.__name__}, a server with exactly the bug this rule exists to "
            "catch: a rule must return a sentence describing the violation")
    also = globals()["rule_stream_is_a_stream"](demo_server_factory(FaultStreamReordered))
    assert also, ("rule_stream_is_a_stream passed a stream in which the terminal status is not the last "
                  "event: check what comes after the terminal update, not only the content type")
    print("exercise 9 looks right: your four rules pass a conforming server and catch their planted faults")


_try("exercise 9", _check_rules, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

def conformance_matrix() -> dict[str, dict[str, str | None]]:
    """Run the harness on the good server and on every planted fault."""
    matrix = {"your server": run_rules(demo_server_factory())}
    for cls, _ in FAULTS:
        matrix[cls.__name__] = run_rules(demo_server_factory(cls))
    return matrix


def _print_matrix() -> None:
    matrix = conformance_matrix()
    targets = {cls.__name__: rule for cls, rule in FAULTS}
    rules = build_rules()
    print(f"{len(rules)} rules; {len(FAULTS)} planted faults\n")
    print(f"{'server':<34}{'target':<8}{'failures':<10}rules that failed")
    caught = 0
    for name, outcome in matrix.items():
        failed = [rid for rid, verdict in outcome.items() if verdict is not None]
        hit = targets.get(name) in failed
        caught += hit
        print(f"{name:<34}{targets.get(name, '-'):<8}{len(failed):<10}{', '.join(failed) or 'none'}")
    print(f"\nyour server fails {sum(v is not None for v in matrix['your server'].values())} rules; "
          f"{caught} of {len(FAULTS)} faults tripped the rule they were planted for")
    for name in ("FaultOldDialectStates", "FaultNeverBlocks"):
        failed = [rid for rid, verdict in matrix[name].items() if verdict is not None]
        first = matrix[name][failed[0]] if failed else "-"
        print(f"  {name}: {len(failed)} rules failed from one root cause; first symptom:\n    {first}")
    assert not any(v for v in matrix["your server"].values()), "your own server failed the harness"
    assert caught == len(FAULTS), "a planted fault escaped the rule it was planted for"


_try("conformance matrix", _print_matrix,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 9"))

## 14. Delegating to an agent you cannot inspect: what the record shows

Now the measurement that says where A2A sits. Give the same job to the same capability twice.
First **as an A2A agent**: the client sends a message and receives a task, status notes and an
artifact; whatever the agent did inside is its own business. Then **as MCP tools**: the same
two tools published by name, description and JSON Schema (`tools/list`), and the *client* makes
every call itself (`tools/call`), the shape T07-L02 built. The specification puts the split
this way: MCP connects an agent to its tools, A2A connects agents to each other
(`claims.yaml`, `a2a-vs-mcp`, `mcp-what-it-is`).

What the lesson counts, from the wire record of each run and from the remote agent's private
log: how many typed parameters are **published before any call**; how many of the remote
agent's internal lookups can be **recovered from the bytes the client saw**; how many wire
messages the delegation took. A third run makes the A2A agent *chatty* (its progress notes
name the keys it looked up) to show that what a client sees under A2A is the agent's choice,
not something the protocol guarantees. This measures record *shapes* in a simulation with
a scripted agent; it says nothing about how a real remote agent behaves.

In [ ]:
class ToolWire:
    """The same capability exposed as MCP tools: `tools/list` publishes name, description and
    JSON Schema; `tools/call` runs one tool. The caller makes every call. Minimal in-memory
    JSON-RPC in the shapes of T07-L02 (MCP 2025-11-25, claims.yaml `mcp-tools-list-shape`)."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.transcript: list[tuple[str, bytes]] = []
        self.calls: list[tuple[str, dict]] = []

    def request(self, method: str, params: dict | None = None) -> dict:
        rid = len(self.transcript) // 2 + 1
        self.transcript.append(("client->server", _dumps(
            {"jsonrpc": "2.0", "id": rid, "method": method, "params": params or {}})))
        result = self._serve(method, params or {})
        self.transcript.append(("server->client", _dumps({"jsonrpc": "2.0", "id": rid, "result": result})))
        return result

    def _serve(self, method: str, params: dict) -> dict:
        if method == "tools/list":
            return {"tools": [{"name": name, "description": TOOL_DESCRIPTIONS[name],
                               "inputSchema": self.registry.schemas[name]}
                              for name in self.registry.tools]}
        if method == "tools/call":
            self.calls.append((params["name"], dict(params.get("arguments", {}))))
            outcome = call_tool(self.registry, params["name"], params.get("arguments", {}))
            text = json.dumps(outcome.output) if outcome.ok else str(outcome.error)
            return {"content": [{"type": "text", "text": text}], "isError": not outcome.ok}
        raise ValueError(f"no such method {method!r}")


def mcp_walk(wire: ToolWire, start_key: str) -> int:
    """The client walks the chain ITSELF over tools/call, and returns the value it reaches."""
    wire.request("tools/list")
    key = start_key
    while True:
        out = json.loads(wire.request("tools/call", {"name": "lookup", "arguments": {"key": key}})
                         ["content"][0]["text"])
        if "value" in out:
            wire.request("tools/call", {"name": "submit", "arguments": {"value": out["value"]}})
            return out["value"]
        key = out["next_key"]


def count_schema_properties(document: Any) -> int:
    """How many typed parameters a published document declares: every property name found under
    a JSON Schema `properties` key, anywhere inside it."""
    if isinstance(document, dict):
        own = len(document["properties"]) if isinstance(document.get("properties"), dict) else 0
        return own + sum(count_schema_properties(v) for v in document.values())
    if isinstance(document, list):
        return sum(count_schema_properties(v) for v in document)
    return 0


def keys_visible(transcript: list[tuple[str, bytes]], keys: list[str]) -> int:
    """How many of `keys` occur anywhere in the recorded bytes, in either direction."""
    text = b"\n".join(raw for _, raw in transcript).decode("utf-8")
    return sum(key in text for key in keys)


class Delegation(NamedTuple):
    published_params: int      # typed parameters published before any call
    internal_lookups: int      # lookups the remote side performed (its private log)
    visible_lookups: int       # of those, keys recoverable from the client's wire record
    wire_messages: int         # messages in both directions
    status_updates: int        # status events the client saw (0 where there is no task)
    answer: int | None         # the value the client ended up with
    visible_keys: tuple[str, ...] = ()   # which of the internal lookup keys the record shows


def delegate_a2a(verbose: bool) -> Delegation:
    server, transport, client = new_run(verbose=verbose)
    events = list(client.stream("SendStreamingMessage", {"message": client.user_message(
        f"solve the chain from {_DEMO_TASK.start_key}")}))
    view = new_view()
    for event in events:
        apply_stream_event(view, event)
    rec = next(iter(server.tasks.values()))
    keys = [args["key"] for name, args in rec.job.calls if name == "lookup"]
    answer_text = next(iter(view["artifacts"].values()))["text"]
    record = b"\n".join(raw for _, raw in transport.transcript).decode("utf-8")
    return Delegation(count_schema_properties(server.agent_card()), len(keys),
                      keys_visible(transport.transcript, keys), len(transport.transcript),
                      sum("statusUpdate" in e for e in events), int(answer_text.split()[-1]),
                      tuple(key for key in keys if key in record))


def delegate_mcp() -> Delegation:
    registry = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
    wire = ToolWire(registry)
    value = mcp_walk(wire, _DEMO_TASK.start_key)
    listed = json.loads(wire.transcript[1][1])["result"]
    keys = [args["key"] for name, args in wire.calls if name == "lookup"]
    return Delegation(count_schema_properties(listed), len(keys), keys_visible(wire.transcript, keys),
                      len(wire.transcript), 0, value)


def _delegation_report() -> None:
    quiet, chatty, mcp = delegate_a2a(False), delegate_a2a(True), delegate_mcp()
    rows = (("typed parameters published before any call", "published_params"),
            ("internal lookups performed remotely", "internal_lookups"),
            ("... of them recoverable from the client's record", "visible_lookups"),
            ("wire messages, both directions", "wire_messages"),
            ("status updates the client saw", "status_updates"))
    print(f"{'':<54}{'A2A quiet':>10}{'A2A chatty':>12}{'MCP tools':>11}")
    for label, field in rows:
        print(f"{label:<54}{getattr(quiet, field):>10}{getattr(chatty, field):>12}{getattr(mcp, field):>11}")
    print(f"{'answer the client ended up with':<54}{quiet.answer:>10}{chatty.answer:>12}{mcp.answer:>11}")
    print()
    typed_only = quiet.visible_lookups == 1 and _DEMO_TASK.start_key in quiet.visible_keys
    print(f"Under A2A the client recovered {quiet.visible_lookups} of {quiet.internal_lookups} internal lookups"
          + (" (the start key, which it typed itself)" if typed_only else "")
          + f", and saw {quiet.status_updates} status updates of the agent's own choosing.")
    print(f"The same agent made chatty recovered {chatty.visible_lookups} of {chatty.internal_lookups}: "
          "nothing in the protocol requires it, and nothing forbids it.")
    print(f"Under MCP the client recovered {mcp.visible_lookups} of {mcp.internal_lookups} (it made every call), "
          f"and was handed {mcp.published_params} typed parameters up front against {quiet.published_params} "
          "from the Agent Card.")
    assert quiet.answer == chatty.answer == mcp.answer == _DEMO_TASK.target, "the three runs disagree on the answer"
    assert quiet.visible_lookups < quiet.internal_lookups, "an opaque agent revealed every lookup"
    assert mcp.visible_lookups == mcp.internal_lookups, "the MCP client should see every call it made"
    assert chatty.visible_lookups == chatty.internal_lookups, "a chatty agent should have named every key"


_try("delegation measurement", _delegation_report,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 5", "exercise 7"))

## 15. Self-check

1. A blocking `SendMessage` returns a task in `TASK_STATE_INPUT_REQUIRED`. What is it?
   a) A failure: the agent could not do the job, so the client should start again
   b) A terminal state: the task is over and its history is read-only
   c) An interrupted state: the task is alive and waits for the client, who continues it by
      sending a message with the same `taskId` and `contextId`
   d) A signal to poll `GetTask` until the state changes on its own

2. A client sends a message naming the `taskId` of a task that is already `COMPLETED`. What
   does the specification's rule for a terminal task require of the server?
   a) It answers `UnsupportedOperationError`: a terminal task cannot accept further messages, so
      follow-up work is a new task, in the same `contextId` if the client wants continuity
   b) It reopens the task and moves it back to `TASK_STATE_WORKING`
   c) It answers `TaskNotFoundError`, because a completed task is deleted
   d) It appends the message to the history and answers with the unchanged task

3. A server's Agent Card declares `capabilities.streaming: false` and a client calls
   `SendStreamingMessage` anyway. What MUST the server do?
   a) Serve a stream: capabilities in the card are only advice
   b) Fall back to a single JSON answer with the finished task
   c) Close the connection without a JSON-RPC response
   d) Answer `UnsupportedOperationError`, and create no task

4. Why can an A2A client recover fewer of a remote agent's internal tool calls than an MCP
   client can, in this lesson's measurement?
   a) A2A messages are encrypted, MCP messages are not
   b) The A2A wire has no field for the remote agent's internal calls: the client sees the task,
      its status notes and artifacts, which the agent chooses; under MCP the client makes the
      calls itself
   c) A2A servers are required to hide their internal calls
   d) MCP servers log every call to the client after the fact

5. Which statement about the two protocols does the A2A documentation itself make?
   a) A2A replaces MCP: agents should stop exposing tools
   b) MCP is for agents talking to agents; A2A is for agents talking to tools
   c) They are the same protocol at two versions
   d) They are complementary: MCP connects an agent to its own tools, A2A connects agents to
      each other

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    def shown(funcs: tuple[str, ...]) -> str:
        text = ", ".join(funcs)
        return text if len(text) <= 46 else text[:43] + "..."

    width = max(len(shown(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {shown(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## 16. What is out of scope, and why

This lesson tests the JSON-RPC binding's core loop. Left out, all specified by A2A 1.0 and each
a lesson of its own: `SubscribeToTask` (re-attach to a running task), push notifications
(webhooks), `ListTasks` with pagination, the authenticated extended Agent Card, Agent Card
signatures (JWS), extensions, multi-tenant routing, the gRPC and HTTP+JSON bindings, real HTTP
and Server-Sent Events, all authentication and authorisation (`AUTH_REQUIRED` exists here only
as a state), file and structured-data parts, and an agent that answers with a direct `Message`
instead of a task (your client and stream fold accept one; the server never sends one). The
state-transition table of exercise 2 is this lesson's reading, not the specification's; and where
the specification lets an agent choose, the harness is silent by design.

## What you built

A validator for the document an agent publishes about itself, a lifecycle that refuses illegal
moves, an A2A server whose handlers respect the rules a third party can test, a client that can
poll, stream and answer a question, and a conformance harness that, run against
deliberately broken servers, tells you which rule each one breaks and how many other rules that
one bug drags down with it. None of it depended on a language model, or on Python at the other
end of the wire: `A2AClient` and the rules only ever see JSON-RPC messages and SSE events, so
they would speak to any server that follows A2A 1.0 over the JSON-RPC binding.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now, so the board is current even if you edited an exercise and did not re-run its cell.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_validate_agent_card, ()),
        ("exercise 2", _check_check_transition, ()),
        ("exercise 3", _check_send_message, ("exercise 2",)),
        ("exercise 4", _check_get_and_cancel, ("exercise 2", "exercise 3")),
        ("exercise 5", _check_stream_message, ("exercise 2", "exercise 3")),
        ("exercise 6", _check_poll, ()),
        ("exercise 7", _check_fold, ()),
        ("exercise 8", _check_converse, ()),
        ("exercise 9", _check_rules,
         ("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5")),
        ("first run", _first_run, ("exercise 1", "exercise 2", "exercise 3")),
        ("polling demo", _poll_demo, ("exercise 2", "exercise 3", "exercise 4", "exercise 6")),
        ("streaming demo", _stream_demo, ("exercise 2", "exercise 3", "exercise 5", "exercise 7")),
        ("conversation demo", _converse_demo, ("exercise 2", "exercise 3", "exercise 8")),
        ("conformance matrix", _print_matrix,
         ("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 9")),
        ("delegation measurement", _delegation_report,
         ("exercise 1", "exercise 2", "exercise 3", "exercise 5", "exercise 7")),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.